# DTCR — Digital-Twin-Enabled Cyber-Resilience Framework
### Algorithm, visualizations and report — Google Colab notebook

Reproduces the analysis of the manuscript *Digital-Twin-Enabled Cyber-Resilience
Framework for Secure Edge-Cloud Orchestration and Data Integrity in Distributed
Smart-Region Infrastructure* from the accompanying deposit. Runs top to bottom in
Google Colab with no local setup.

**It does three things:**
1. installs the deposit and its dependencies;
2. runs the full pipeline — reference dataset → statistics → NRI → figures;
3. renders every figure inline and prints a self-contained report.

> ⚠️ **The `data/` directory is a *synthetic reference dataset*, not
> measurement.** Every value it produces is consistent with the manuscript by
> construction, but nothing here is an experimental result. See `PROVENANCE.md`.
> To reproduce the *real* results, replace `data/` with measurement exports in
> the schema of `DATA_DICTIONARY.md` and re-run — no code changes needed.

## 1. Setup

By default the notebook writes a **self-contained copy of the deposit** into the
Colab runtime, so it runs even without network access. If you have pushed the
deposit to GitHub, set `GIT_URL` below to clone it instead.

In [ ]:
# If you have the deposit on GitHub, put its URL here to clone it.
# Leave as None to bootstrap a self-contained copy embedded in this notebook.
GIT_URL = None          # e.g. "https://github.com/<owner>/<repo>.git"
SUBDIR  = "publication/dtcr-deposit"   # path to the deposit inside the repo

import os, sys, subprocess, pathlib

def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-2000:])

sh("pip -q install numpy pandas scipy matplotlib pyyaml tabulate")

if GIT_URL:
    sh(f"git clone --depth 1 {GIT_URL} _repo")
    ROOT = pathlib.Path("_repo")/SUBDIR
else:
    ROOT = pathlib.Path("dtcr-deposit")
    (ROOT/"analysis"/"dtcr").mkdir(parents=True, exist_ok=True)
    (ROOT/"configs").mkdir(parents=True, exist_ok=True)
    print("No GIT_URL set: the next cells write a self-contained copy into", ROOT)
ROOT = ROOT.resolve()   # absolute, so later os.chdir(ROOT) does not break relative lookups
print("deposit root:", ROOT)

## 2. Bootstrap the reference library (skipped when cloning)

Writes `analysis/dtcr/` — the code is identical to the deposit; it is embedded so
the notebook is runnable on its own.

In [ ]:
if not GIT_URL:
    pkg = ROOT/'analysis'/'dtcr'
    (pkg/'__init__.py').write_text("__version__='1.0.0'\n")
    FILES = {}
    FILES['audit'+'.py'] = '"""Probabilistic block audit (manuscript Eq. 4-5).\n\nA replica holds ``l`` blocks of which ``d`` are corrupted.  ``r`` distinct blocks\nare challenged without replacement.  ``p_detect_exact`` implements the\nhypergeometric probability of hitting at least one corrupted block;\n``p_detect_bound`` implements the independent-sampling lower bound; and\n``r_min`` inverts the bound to a sufficient challenge budget.\n"""\nfrom __future__ import annotations\n\nimport math\n\n__all__ = ["p_detect_exact", "p_detect_bound", "r_min", "challenge_table"]\n\n\ndef p_detect_exact(l: int, d: int, r: int) -> float:\n    """Exact without-replacement detection probability, Eq. (4).\n\n    P_det = 1 - prod_{j=0}^{r-1} (l - d - j) / (l - j)\n    """\n    if not (0 <= d <= l):\n        raise ValueError("require 0 <= d <= l")\n    if not (0 <= r <= l):\n        raise ValueError("require 0 <= r <= l")\n    if d == 0:\n        return 0.0\n    if r > l - d:  # more challenges than clean blocks -> certain hit\n        return 1.0\n    miss = 1.0\n    for j in range(r):\n        miss *= (l - d - j) / (l - j)\n    return 1.0 - miss\n\n\ndef p_detect_bound(p: float, r: int) -> float:\n    """Conservative lower bound 1 - (1 - p)^r with p = d/l."""\n    if not (0.0 <= p <= 1.0):\n        raise ValueError("require 0 <= p <= 1")\n    return 1.0 - (1.0 - p) ** r\n\n\ndef r_min(p: float, eta: float) -> int:\n    """Smallest challenge count meeting a lower-bound target eta, Eq. (5)."""\n    if not (0.0 < p < 1.0):\n        raise ValueError("require 0 < p < 1")\n    if not (0.0 < eta < 1.0):\n        raise ValueError("require 0 < eta < 1")\n    return int(math.ceil(math.log(1.0 - eta) / math.log(1.0 - p)))\n\n\ndef challenge_table(fractions=(0.01, 0.05, 0.10, 0.20),\n                    targets=(0.90, 0.95, 0.99),\n                    l: int = 10000):\n    """Reproduce manuscript Table 5 and add the exact probability achieved."""\n    rows = []\n    for p in fractions:\n        row = {"corrupted_fraction": p}\n        for eta in targets:\n            r = r_min(p, eta)\n            row[f"r_min_{int(eta * 100)}"] = r\n            row[f"p_exact_{int(eta * 100)}"] = p_detect_exact(l, int(round(p * l)), r)\n        rows.append(row)\n    return rows\n'
    FILES['trust'+'.py'] = '"""Dynamic provenance-aware trust (manuscript Eq. 2-3)."""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\n__all__ = ["TrustWeights", "instantaneous_trust", "smooth_trust", "TrustTracker"]\n\n\n@dataclass(frozen=True)\nclass TrustWeights:\n    """Weights of the integrity / provenance / behaviour components; must sum to 1."""\n\n    alpha: float = 0.40\n    beta: float = 0.35\n    gamma: float = 0.25\n\n    def __post_init__(self) -> None:\n        total = self.alpha + self.beta + self.gamma\n        if abs(total - 1.0) > 1e-9:\n            raise ValueError(f"alpha+beta+gamma must equal 1, got {total}")\n\n\ndef instantaneous_trust(c: float, q: float, b: float,\n                        w: TrustWeights | None = None) -> float:\n    """T_i^inst(t) = alpha*c + beta*q + gamma*b, Eq. (2)."""\n    w = w or TrustWeights()\n    return w.alpha * c + w.beta * q + w.gamma * b\n\n\ndef smooth_trust(prev: float, inst: float, rho: float = 0.60) -> float:\n    """T_i(t) = rho*T_i(t-1) + (1-rho)*T_i^inst(t), Eq. (3)."""\n    if not (0.0 <= rho < 1.0):\n        raise ValueError("require 0 <= rho < 1")\n    return rho * prev + (1.0 - rho) * inst\n\n\nclass TrustTracker:\n    """Stateful per-asset trust with an explicit initial value."""\n\n    def __init__(self, initial: float = 1.0, rho: float = 0.60,\n                 weights: TrustWeights | None = None):\n        self.value = float(initial)\n        self.rho = rho\n        self.weights = weights or TrustWeights()\n        self.history = [self.value]\n\n    def update(self, c: float, q: float, b: float) -> float:\n        inst = instantaneous_trust(c, q, b, self.weights)\n        self.value = smooth_trust(self.value, inst, self.rho)\n        self.history.append(self.value)\n        return self.value\n'
    FILES['anomaly'+'.py'] = '"""Anomaly scoring (manuscript Eq. 6-7).\n\nThe manuscript maps the squared Mahalanobis distance through\n``a = 1 - exp(-d^2 / 2)``, which is a bounded monotone score but is *not* the\nprobability of anomaly for a p-dimensional feature vector.  Under the Gaussian\nassumption stated in Section 2.4 the calibrated quantity is the chi-square CDF\nwith p degrees of freedom.  Both mappings are implemented so that the printed\nmanuscript value and the corrected value can be reported side by side.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom scipy import stats as _sps\n\n__all__ = [\n    "BaselineModel",\n    "mahalanobis_sq",\n    "score_legacy",\n    "score_chi2",\n    "empirical_calibration",\n    "gaussian_fit_check",\n]\n\n\nclass BaselineModel:\n    """Normal-phase mean and regularised covariance for one asset.\n\n    Parameters\n    ----------\n    shrinkage\n        Ridge applied as ``Sigma + shrinkage * tr(Sigma)/p * I``.  A strictly\n        positive value is required whenever ``n_samples`` is not much larger than\n        the feature dimension, otherwise ``Sigma`` is ill-conditioned and the\n        distances are not comparable across assets.\n    """\n\n    def __init__(self, X: np.ndarray, shrinkage: float = 0.05):\n        X = np.asarray(X, dtype=float)\n        if X.ndim != 2:\n            raise ValueError("X must be (n_samples, n_features)")\n        self.n_samples, self.p = X.shape\n        if self.n_samples <= self.p:\n            raise ValueError(\n                f"n_samples ({self.n_samples}) must exceed feature dimension ({self.p})"\n            )\n        self.mu = X.mean(axis=0)\n        cov = np.atleast_2d(np.cov(X, rowvar=False))\n        self.shrinkage = float(shrinkage)\n        self.sigma = cov + self.shrinkage * np.trace(cov) / self.p * np.eye(self.p)\n        self.sigma_inv = np.linalg.inv(self.sigma)\n        self.condition_number = float(np.linalg.cond(self.sigma))\n\n    def distance_sq(self, z: np.ndarray):\n        return mahalanobis_sq(z, self.mu, self.sigma_inv)\n\n\ndef mahalanobis_sq(z, mu, sigma_inv):\n    """d_i^2(t) = (z - mu)^T Sigma^-1 (z - mu), Eq. (6)."""\n    z = np.atleast_2d(np.asarray(z, dtype=float))\n    delta = z - np.asarray(mu, dtype=float)\n    d2 = np.einsum("ij,jk,ik->i", delta, sigma_inv, delta)\n    return d2 if d2.size > 1 else float(d2[0])\n\n\ndef score_legacy(d2):\n    """Manuscript Eq. (7) as printed: a = 1 - exp(-d^2/2). A monotone score only."""\n    return 1.0 - np.exp(-0.5 * np.asarray(d2, dtype=float))\n\n\ndef score_chi2(d2, p: int):\n    """Corrected calibration a = F_{chi2_p}(d^2); a probability under Gaussianity."""\n    return _sps.chi2.cdf(np.asarray(d2, dtype=float), df=p)\n\n\ndef empirical_calibration(d2_calibration, d2_test):\n    """Distribution-free alternative: empirical CDF of the calibration distances.\n\n    Use when the Gaussian assumption fails the goodness-of-fit check; the result\n    is a calibrated tail probability rather than an unnormalised score.\n    """\n    ref = np.sort(np.asarray(d2_calibration, dtype=float))\n    return np.searchsorted(ref, np.asarray(d2_test, dtype=float), side="right") / ref.size\n\n\ndef gaussian_fit_check(d2, p: int):\n    """Kolmogorov-Smirnov test of d^2 against chi2_p on the calibration set.\n\n    Reported in the manuscript so the Gaussian assumption behind Eq. (7) is\n    verified rather than assumed.\n    """\n    d2 = np.asarray(d2, dtype=float)\n    ks = _sps.kstest(d2, "chi2", args=(p,))\n    return {"n": int(d2.size), "df": p, "ks_statistic": float(ks.statistic),\n            "p_value": float(ks.pvalue)}\n'
    FILES['risk'+'.py'] = '"""Local risk and dependency-graph risk propagation (manuscript Eq. 8-11).\n\nTwo local-risk aggregations are provided.  ``local_risk_product`` is the\nmultiplicative form printed in the manuscript; it implements a hard AND and\ncollapses to zero whenever any single factor is zero.  ``local_risk_additive``\nis the weighted alternative whose weights are chosen by ROC analysis on the\ncalibration set.  ``select_aggregation`` performs that comparison so the choice\nis empirical rather than asserted.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\n__all__ = [\n    "local_risk_product",\n    "local_risk_additive",\n    "row_normalise",\n    "spectral_radius",\n    "convergence_margin",\n    "propagate",\n    "propagate_iterative",\n    "amplification",\n    "select_aggregation",\n]\n\n\ndef local_risk_product(a, T, s):\n    """R_i = a_i (1 - T_i) s_i, Eq. (8) as printed."""\n    return np.asarray(a, float) * (1.0 - np.asarray(T, float)) * np.asarray(s, float)\n\n\ndef local_risk_additive(a, T, s, w_a=0.45, w_t=0.35, w_at=0.20):\n    """R_i = s_i [ w_a a_i + w_t (1 - T_i) + w_at a_i (1 - T_i) ], weights sum to 1."""\n    total = w_a + w_t + w_at\n    if abs(total - 1.0) > 1e-9:\n        raise ValueError(f"weights must sum to 1, got {total}")\n    a = np.asarray(a, float)\n    t = 1.0 - np.asarray(T, float)\n    return np.asarray(s, float) * (w_a * a + w_t * t + w_at * a * t)\n\n\ndef row_normalise(W: np.ndarray) -> np.ndarray:\n    """Normalise each column of W to unit outgoing influence mass.\n\n    W[i, j] is the weight of the dependency of j on i, so the propagation\n    operator is W^T.  Normalising the outgoing mass of every source keeps the\n    spectral radius bounded by 1 and makes lambda the only tuning knob.\n    """\n    W = np.asarray(W, dtype=float).copy()\n    out = W.sum(axis=1, keepdims=True)\n    nz = out.squeeze(-1) > 0\n    W[nz] = W[nz] / out[nz]\n    return W\n\n\ndef spectral_radius(W: np.ndarray, lam: float) -> float:\n    """rho(lambda W^T)."""\n    eig = np.linalg.eigvals(lam * np.asarray(W, float).T)\n    return float(np.max(np.abs(eig)))\n\n\ndef convergence_margin(W: np.ndarray, lam: float) -> float:\n    """1 - rho(lambda W^T); must be strictly positive for Eq. (10) to hold."""\n    return 1.0 - spectral_radius(W, lam)\n\n\ndef propagate(R, W, lam: float):\n    """Closed form R~ = (I - lambda W^T)^-1 R, Eq. (10), with a convergence guard."""\n    R = np.asarray(R, float)\n    W = np.asarray(W, float)\n    margin = convergence_margin(W, lam)\n    if margin <= 0:\n        raise ValueError(\n            f"rho(lambda W^T) = {1 - margin:.4f} >= 1; Eq. (10) does not converge"\n        )\n    return np.linalg.solve(np.eye(W.shape[0]) - lam * W.T, R)\n\n\ndef propagate_iterative(R, W, lam: float, iters: int = 200):\n    """Fixed-point iteration of Eq. (9); used to verify the closed form."""\n    R = np.asarray(R, float)\n    W = np.asarray(W, float)\n    x = R.copy()\n    for _ in range(iters):\n        x = R + lam * W.T @ x\n    return x\n\n\ndef amplification(R, R_tilde) -> float:\n    """kappa = ||R~||_1 / ||R||_1, Eq. (11); returns 1.0 when ||R||_1 = 0."""\n    denom = float(np.abs(np.asarray(R, float)).sum())\n    if denom == 0.0:\n        return 1.0  # no local risk anywhere: define no amplification\n    return float(np.abs(np.asarray(R_tilde, float)).sum()) / denom\n\n\ndef _auc(scores, labels) -> float:\n    """Rank-based AUC without external dependencies."""\n    scores = np.asarray(scores, float)\n    labels = np.asarray(labels, int)\n    pos, neg = labels == 1, labels == 0\n    if pos.sum() == 0 or neg.sum() == 0:\n        return float("nan")\n    order = np.argsort(scores, kind="mergesort")\n    ranks = np.empty_like(order, dtype=float)\n    ranks[order] = np.arange(1, scores.size + 1, dtype=float)\n    # average ranks for ties\n    uniq, inv, counts = np.unique(scores, return_inverse=True, return_counts=True)\n    mean_rank = np.zeros(uniq.size)\n    np.add.at(mean_rank, inv, ranks)\n    mean_rank /= counts\n    ranks = mean_rank[inv]\n    n_pos, n_neg = pos.sum(), neg.sum()\n    return float((ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))\n\n\ndef select_aggregation(a, T, s, labels, weight_grid=None):\n    """Compare Eq. (8) against the additive form by AUC on labelled data.\n\n    Returns the AUC of each candidate and the selected form, so the manuscript\n    reports an empirically chosen aggregation instead of a postulated one.\n    """\n    weight_grid = weight_grid or [\n        (0.45, 0.35, 0.20), (0.34, 0.33, 0.33), (0.60, 0.30, 0.10), (0.30, 0.50, 0.20)\n    ]\n    prod = local_risk_product(a, T, s)\n    result = {"product_auc": _auc(prod, labels), "additive": []}\n    best = ("product", result["product_auc"], None)\n    for w in weight_grid:\n        add = local_risk_additive(a, T, s, *w)\n        auc = _auc(add, labels)\n        result["additive"].append({"weights": w, "auc": auc})\n        if auc > best[1]:\n            best = ("additive", auc, w)\n    result["selected_form"] = best[0]\n    result["selected_auc"] = best[1]\n    result["selected_weights"] = best[2]\n    return result\n'
    FILES['orchestration'+'.py'] = '"""Policy-constrained security orchestration (manuscript Eq. 12-13, Algorithm 1).\n\nThe manuscript prints a finite penalty ``P_viol`` inside the objective while\nAlgorithm 1 states that policy-violating candidates are rejected outright.  This\nimplementation resolves the inconsistency in favour of Algorithm 1: admissibility\nis a hard constraint evaluated before scoring, and the objective contains only\nthe risk, overhead and disruption terms with the three separate coefficients\nmu_1, mu_2, mu_3 used consistently.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Sequence\n\nimport numpy as np\n\n__all__ = ["ResourceVector", "Node", "Workload", "Candidate", "Objective",\n           "admissible", "select_action"]\n\n\n@dataclass(frozen=True)\nclass ResourceVector:\n    """Vector capacity/demand; the manuscript\'s scalar C_j is generalised here."""\n\n    cpu: float = 0.0\n    ram: float = 0.0\n    storage: float = 0.0\n    network: float = 0.0\n\n    def __le__(self, other: "ResourceVector") -> bool:\n        return (self.cpu <= other.cpu and self.ram <= other.ram\n                and self.storage <= other.storage and self.network <= other.network)\n\n    def __add__(self, other: "ResourceVector") -> "ResourceVector":\n        return ResourceVector(self.cpu + other.cpu, self.ram + other.ram,\n                              self.storage + other.storage, self.network + other.network)\n\n\n@dataclass\nclass Node:\n    node_id: str\n    capacity: ResourceVector\n    used: ResourceVector = field(default_factory=ResourceVector)\n    security_label: int = 0      # lattice level; higher dominates\n    trust: float = 1.0\n    domain: str = "d0"\n\n    def free(self) -> ResourceVector:\n        return ResourceVector(self.capacity.cpu - self.used.cpu,\n                              self.capacity.ram - self.used.ram,\n                              self.capacity.storage - self.used.storage,\n                              self.capacity.network - self.used.network)\n\n\n@dataclass\nclass Workload:\n    workload_id: str\n    demand: ResourceVector\n    security_label: int = 0\n    min_host_trust: float = 0.0   # tau_i\n    allowed_domains: tuple = ()   # empty tuple means "no cross-domain restriction"\n\n\n@dataclass\nclass Candidate:\n    """A candidate protective action with its twin-predicted consequences."""\n\n    action: str\n    residual_risk: float          # sum_i R~_i(t | x)\n    overhead_cpu: float           # O_cpu(x), normalised to [0, 1]\n    overhead_net: float           # O_net(x), normalised to [0, 1]\n    disruption: float             # D(x), normalised to [0, 1]\n    placement: dict = field(default_factory=dict)   # workload_id -> node_id\n\n\n@dataclass(frozen=True)\nclass Objective:\n    """Coefficients of Eq. (12). All terms are dimensionless in [0, 1]."""\n\n    mu1: float = 0.20   # compute overhead\n    mu2: float = 0.15   # communication overhead\n    mu3: float = 0.25   # service disruption\n\n    def value(self, c: Candidate) -> float:\n        return (c.residual_risk + self.mu1 * c.overhead_cpu\n                + self.mu2 * c.overhead_net + self.mu3 * c.disruption)\n\n\ndef admissible(candidate: Candidate, nodes: dict, workloads: dict):\n    """Evaluate the hard constraints of Eq. (13); returns (bool, reasons).\n\n    Constraints, in the order checked:\n      1. every workload is placed exactly once;\n      2. vector capacity is respected on every node;\n      3. the node security label dominates the workload label;\n      4. the host trust meets the workload\'s minimum tau_i;\n      5. the placement respects the workload\'s admissible domains.\n    """\n    reasons = []\n    placed = candidate.placement\n    for wid in workloads:\n        if placed.get(wid) is None:\n            reasons.append(f"workload {wid} unplaced")\n    load = {nid: ResourceVector() for nid in nodes}\n    for wid, nid in placed.items():\n        if nid not in nodes:\n            reasons.append(f"unknown node {nid}")\n            continue\n        load[nid] = load[nid] + workloads[wid].demand\n        w, n = workloads[wid], nodes[nid]\n        if n.security_label < w.security_label:\n            reasons.append(f"label violation: {wid}@{nid}")\n        if n.trust < w.min_host_trust:\n            reasons.append(\n                f"trust violation: {wid} needs {w.min_host_trust:.2f}, {nid} has {n.trust:.2f}")\n        if w.allowed_domains and n.domain not in w.allowed_domains:\n            reasons.append(f"domain violation: {wid} -> {n.domain}")\n    for nid, used in load.items():\n        n = nodes[nid]\n        total = used + n.used\n        if not (total <= n.capacity):\n            reasons.append(f"capacity violation on {nid}")\n    return (len(reasons) == 0), reasons\n\n\ndef select_action(candidates: Sequence[Candidate], nodes: dict, workloads: dict,\n                  objective: Objective | None = None, tie_epsilon: float = 1e-9):\n    """Algorithm 1 steps 7-9: reject inadmissible candidates, then argmin J(x).\n\n    Ties within ``tie_epsilon`` are broken by lowest disruption, then lowest\n    compute overhead, then lexicographic action name, so the selection is\n    deterministic and reproducible across runs.\n    """\n    objective = objective or Objective()\n    scored, rejected = [], []\n    for c in candidates:\n        ok, reasons = admissible(c, nodes, workloads)\n        if ok:\n            scored.append((objective.value(c), c))\n        else:\n            rejected.append({"action": c.action, "reasons": reasons})\n    if not scored:\n        return {"selected": None, "rejected": rejected, "ranking": []}\n    best = min(s for s, _ in scored)\n    tied = [c for s, c in scored if s <= best + tie_epsilon]\n    tied.sort(key=lambda c: (c.disruption, c.overhead_cpu, c.action))\n    ranking = sorted(({"action": c.action, "J": s} for s, c in scored),\n                     key=lambda r: r["J"])\n    return {"selected": tied[0], "objective_value": best,\n            "rejected": rejected, "ranking": ranking}\n'
    FILES['resilience'+'.py'] = '"""Availability, recovery time and the normalized resilience index.\n\nImplements manuscript Eq. (15), (18) and (19).  The manuscript reuses the symbol\n``t_d`` for both detection time and disruption time; this module keeps them\nstrictly separate as ``t_det`` and ``t_dis`` and the integration window of\nEq. (18) is anchored on ``t_dis``.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\n\n__all__ = ["NRIConfig", "recovery_time", "nri", "nri_from_trace", "resilience_deficit"]\n\n\n@dataclass(frozen=True)\nclass NRIConfig:\n    """Every quantity Figure 6 and Eq. (18) depend on, stated explicitly."""\n\n    rto: float = 300.0            # organizational recovery-time objective, s\n    a_min: float = 0.95           # acceptable availability threshold\n    a_max: float = 1.00           # ideal availability used as the denominator\n    hold: float = 30.0            # Delta_h, s\n    sampling_interval: float = 1.0  # s\n\n\ndef recovery_time(t: np.ndarray, a: np.ndarray, t_attack: float,\n                  cfg: NRIConfig) -> float:\n    """Eq. (15): first instant from which A >= a_min holds for the whole hold window.\n\n    Returns ``nan`` when availability never satisfies the hold condition inside\n    the observed trace; the caller must treat such runs as censored rather than\n    dropping them silently.\n    """\n    t = np.asarray(t, float)\n    a = np.asarray(a, float)\n    mask = t >= t_attack\n    ts, as_ = t[mask], a[mask]\n    if ts.size == 0:\n        return float("nan")\n    ok = as_ >= cfg.a_min\n    for i in range(ts.size):\n        if not ok[i]:\n            continue\n        window = (ts >= ts[i]) & (ts <= ts[i] + cfg.hold)\n        if ts[window][-1] < ts[i] + cfg.hold - cfg.sampling_interval / 2:\n            break  # hold window runs past the end of the trace -> censored\n        if ok[window].all():\n            return float(ts[i] - t_attack)\n    return float("nan")\n\n\ndef nri(t: np.ndarray, a: np.ndarray, t_dis: float, cfg: NRIConfig) -> float:\n    """Eq. (18)-(19): trapezoidal AUC over [t_dis, t_dis + 2*RTO] normalised by A_max.\n\n    The window is resampled onto the trace grid with linear interpolation at both\n    endpoints so that traces with different sampling offsets are comparable.\n    """\n    t = np.asarray(t, float)\n    a = np.asarray(a, float)\n    t0, t1 = t_dis, t_dis + 2.0 * cfg.rto\n    if t1 > t[-1] + 1e-9:\n        raise ValueError(\n            f"trace ends at {t[-1]:.1f}s but the NRI window needs {t1:.1f}s")\n    grid = np.unique(np.concatenate(([t0], t[(t > t0) & (t < t1)], [t1])))\n    vals = np.interp(grid, t, a)\n    area = np.trapezoid(vals, grid) if hasattr(np, "trapezoid") else np.trapz(vals, grid)\n    return float(area / (cfg.a_max * (t1 - t0)))\n\n\ndef nri_from_trace(df, cfg: NRIConfig, t_col="t_s", a_col="availability",\n                   t_dis: float | None = None) -> float:\n    """Convenience wrapper for a two-column availability trace DataFrame."""\n    t = df[t_col].to_numpy(float)\n    a = df[a_col].to_numpy(float)\n    if t_dis is None:\n        below = np.flatnonzero(a < cfg.a_min)\n        if below.size == 0:\n            return 1.0\n        t_dis = float(t[below[0]])\n    return nri(t, a, t_dis, cfg)\n\n\ndef resilience_deficit(value: float) -> float:\n    """1 - NRI, the cumulative availability loss relative to the ideal curve."""\n    return 1.0 - float(value)\n'
    FILES['stats'+'.py'] = '"""Statistical estimators for every effect reported in the manuscript.\n\nCovers the descriptive statistics, paired and unpaired inference, bootstrap\nconfidence intervals for skewed latency distributions, effect sizes, and\nbinomial intervals for classification rates required by Sections 2.3, 6.1 and\n9 of the revision plan.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom scipy import stats as sps\n\n__all__ = ["describe", "t_ci", "bootstrap_ci", "paired_bootstrap_diff_ci",\n           "hedges_g", "cliffs_delta", "compare_groups", "wilson_ci",\n           "classification_metrics", "holm_bonferroni"]\n\n\ndef describe(x) -> dict:\n    """n, mean, SD, median, IQR, min, max and the 95% t interval for the mean."""\n    x = np.asarray(x, float)\n    x = x[~np.isnan(x)]\n    n = x.size\n    if n == 0:\n        return {"n": 0}\n    sd = float(x.std(ddof=1)) if n > 1 else float("nan")\n    q1, q3 = np.percentile(x, [25, 75])\n    lo, hi = t_ci(x)\n    return {"n": int(n), "mean": float(x.mean()), "sd": sd,\n            "median": float(np.median(x)), "q1": float(q1), "q3": float(q3),\n            "iqr": float(q3 - q1), "min": float(x.min()), "max": float(x.max()),\n            "ci95_lo": lo, "ci95_hi": hi}\n\n\ndef t_ci(x, conf: float = 0.95):\n    """Eq. (23): mean +/- t_{1-alpha/2, n-1} * s / sqrt(n)."""\n    x = np.asarray(x, float)\n    x = x[~np.isnan(x)]\n    n = x.size\n    if n < 2:\n        return (float("nan"), float("nan"))\n    h = sps.t.ppf(0.5 + conf / 2, n - 1) * x.std(ddof=1) / np.sqrt(n)\n    return (float(x.mean() - h), float(x.mean() + h))\n\n\ndef bootstrap_ci(x, statistic=np.mean, n_boot: int = 10000, conf: float = 0.95,\n                 seed: int = 20260731):\n    """Percentile bootstrap interval; preferred for skewed latency distributions."""\n    x = np.asarray(x, float)\n    x = x[~np.isnan(x)]\n    rng = np.random.default_rng(seed)\n    idx = rng.integers(0, x.size, size=(n_boot, x.size))\n    boots = statistic(x[idx], axis=1)\n    lo, hi = np.percentile(boots, [(1 - conf) / 2 * 100, (1 + conf) / 2 * 100])\n    return {"point": float(statistic(x)), "ci95_lo": float(lo), "ci95_hi": float(hi),\n            "n_boot": n_boot}\n\n\ndef paired_bootstrap_diff_ci(a, b, n_boot: int = 10000, conf: float = 0.95,\n                             seed: int = 20260731):\n    """Bootstrap interval for the paired mean difference a - b."""\n    a, b = np.asarray(a, float), np.asarray(b, float)\n    if a.shape != b.shape:\n        raise ValueError("paired samples must have equal length")\n    keep = ~(np.isnan(a) | np.isnan(b))\n    a, b = a[keep], b[keep]\n    rng = np.random.default_rng(seed)\n    idx = rng.integers(0, a.size, size=(n_boot, a.size))\n    boots = (a[idx] - b[idx]).mean(axis=1)\n    lo, hi = np.percentile(boots, [(1 - conf) / 2 * 100, (1 + conf) / 2 * 100])\n    return {"mean_diff": float((a - b).mean()), "ci95_lo": float(lo),\n            "ci95_hi": float(hi), "n_pairs": int(a.size)}\n\n\ndef hedges_g(a, b) -> float:\n    """Bias-corrected standardised mean difference for two independent samples."""\n    a, b = np.asarray(a, float), np.asarray(b, float)\n    a, b = a[~np.isnan(a)], b[~np.isnan(b)]\n    na, nb = a.size, b.size\n    if na < 2 or nb < 2:\n        return float("nan")\n    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))\n    if sp == 0:\n        return float("nan")\n    d = (a.mean() - b.mean()) / sp\n    j = 1 - 3 / (4 * (na + nb) - 9)\n    return float(d * j)\n\n\ndef cliffs_delta(a, b) -> float:\n    """Non-parametric effect size in [-1, 1]; robust to the skew of recovery times."""\n    a, b = np.asarray(a, float), np.asarray(b, float)\n    a, b = a[~np.isnan(a)], b[~np.isnan(b)]\n    if a.size == 0 or b.size == 0:\n        return float("nan")\n    gt = (a[:, None] > b[None, :]).sum()\n    lt = (a[:, None] < b[None, :]).sum()\n    return float((gt - lt) / (a.size * b.size))\n\n\ndef compare_groups(baseline, framework, paired: bool = True, seed: int = 20260731):\n    """Full comparison record for one scenario and one lower-is-better metric.\n\n    ``paired=True`` is correct only when the two arms ran on a single\n    interleaved schedule with matched repetition indices; the flag is recorded in\n    the output so the manuscript states which design was used.\n    """\n    b = np.asarray(baseline, float)\n    f = np.asarray(framework, float)\n    out = {"baseline": describe(b), "framework": describe(f), "paired": bool(paired)}\n    out["abs_diff_mean"] = out["framework"]["mean"] - out["baseline"]["mean"]\n    out["rel_reduction_pct"] = (\n        100.0 * (out["baseline"]["mean"] - out["framework"]["mean"]) / out["baseline"]["mean"]\n        if out["baseline"]["mean"] else float("nan"))\n    if paired and b.size == f.size:\n        keep = ~(np.isnan(b) | np.isnan(f))\n        tt = sps.ttest_rel(b[keep], f[keep])\n        wx = sps.wilcoxon(b[keep], f[keep]) if keep.sum() > 0 else None\n        out["test"] = {"name": "paired t-test", "statistic": float(tt.statistic),\n                       "p_value": float(tt.pvalue), "df": int(keep.sum() - 1)}\n        out["test_nonparametric"] = {\n            "name": "Wilcoxon signed-rank",\n            "statistic": float(wx.statistic), "p_value": float(wx.pvalue)}\n        out["diff_ci"] = paired_bootstrap_diff_ci(f, b, seed=seed)\n    else:\n        tt = sps.ttest_ind(b, f, equal_var=False, nan_policy="omit")\n        mw = sps.mannwhitneyu(b[~np.isnan(b)], f[~np.isnan(f)], alternative="two-sided")\n        out["test"] = {"name": "Welch t-test", "statistic": float(tt.statistic),\n                       "p_value": float(tt.pvalue)}\n        out["test_nonparametric"] = {"name": "Mann-Whitney U",\n                                     "statistic": float(mw.statistic),\n                                     "p_value": float(mw.pvalue)}\n    out["hedges_g"] = hedges_g(b, f)\n    out["cliffs_delta"] = cliffs_delta(b, f)\n    return out\n\n\ndef wilson_ci(successes: int, n: int, conf: float = 0.95):\n    """Wilson score interval; correct near 0 and 1 where the normal interval fails."""\n    if n == 0:\n        return (float("nan"), float("nan"))\n    z = sps.norm.ppf(0.5 + conf / 2)\n    p = successes / n\n    denom = 1 + z ** 2 / n\n    centre = (p + z ** 2 / (2 * n)) / denom\n    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / denom\n    return (float(max(0.0, centre - half)), float(min(1.0, centre + half)))\n\n\ndef classification_metrics(tp: int, tn: int, fp: int, fn: int) -> dict:\n    """Full confusion-matrix report with Wilson intervals for every rate."""\n    n = tp + tn + fp + fn\n    pos, neg = tp + fn, tn + fp\n    def _safe(a, b):\n        return a / b if b else float("nan")\n    acc = _safe(tp + tn, n)\n    rec = _safe(tp, pos)\n    spec = _safe(tn, neg)\n    prec = _safe(tp, tp + fp)\n    f1 = _safe(2 * prec * rec, prec + rec) if not (np.isnan(prec) or np.isnan(rec)) else float("nan")\n    mcc_den = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))\n    mcc = (tp * tn - fp * fn) / mcc_den if mcc_den > 0 else float("nan")\n    return {\n        "n": n, "positives": pos, "negatives": neg,\n        "tp": tp, "tn": tn, "fp": fp, "fn": fn,\n        "accuracy": acc, "accuracy_ci95": wilson_ci(tp + tn, n),\n        "recall": rec, "recall_ci95": wilson_ci(tp, pos),\n        "specificity": spec, "specificity_ci95": wilson_ci(tn, neg),\n        "precision": prec, "precision_ci95": wilson_ci(tp, tp + fp),\n        "f1": f1, "mcc": float(mcc),\n        "fpr": _safe(fp, neg), "fpr_ci95": wilson_ci(fp, neg),\n        "fnr": _safe(fn, pos), "fnr_ci95": wilson_ci(fn, pos),\n    }\n\n\ndef holm_bonferroni(pvalues, alpha: float = 0.05):\n    """Holm correction for the family of scenario-level tests."""\n    p = np.asarray(pvalues, float)\n    order = np.argsort(p)\n    m = p.size\n    adjusted = np.empty(m)\n    running = 0.0\n    for rank, idx in enumerate(order):\n        running = max(running, (m - rank) * p[idx])\n        adjusted[idx] = min(1.0, running)\n    return {"p_adjusted": adjusted.tolist(),\n            "reject": (adjusted <= alpha).tolist(), "alpha": alpha}\n'
    for name, text in FILES.items():
        (pkg/name).write_text(text)
    print('wrote dtcr library:', ', '.join(FILES))
else:
    print('cloning from GitHub; skipping embedded library')

## 3. Bootstrap data generator, analysis scripts and config (skipped when cloning)

In [ ]:
if not GIT_URL:
    SCRIPTS = {}
    SCRIPTS['simulate_reference_dataset.py'] = '#!/usr/bin/env python3\n"""Generate the seeded SYNTHETIC reference dataset for the DTCR deposit.\n\nWHAT THIS SCRIPT IS\n-------------------\nThe manuscript reports only aggregate means.  The run-level observations that\nproduced them were not preserved, so this script generates a *synthetic\nreference dataset* whose per-scenario aggregates reproduce the published means\nexactly and whose availability traces, recovery times and NRI values are\nmutually consistent by construction (they are all derived from the same traces\nthrough ``dtcr.resilience``).\n\nWHAT THIS SCRIPT IS NOT\n-----------------------\nIt is NOT a measurement.  No row it writes may be reported as an experimental\nobservation.  Its purpose is to make the analysis pipeline, the figures, the\nstatistics and the manuscript tables executable end to end *before* the real\nruns exist, so that replacing ``data/`` with genuine measurement exports\nreproduces the whole results section without touching a line of analysis code.\nEvery file it writes carries ``data_origin=synthetic_reference`` in a column and\nin its header comment.  ``analysis/verify_repository.py`` refuses to certify a\ndataset that still carries that marker as submission-ready.\n\nUsage\n-----\n    python analysis/simulate_reference_dataset.py --out data\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom dtcr.resilience import NRIConfig, recovery_time, nri  # noqa: E402\n\nDATA_ORIGIN = "synthetic_reference"\nSEED = 20260731\n\nSCENARIOS = ["S1", "S2", "S3", "S4"]\nMETHODS = ["baseline", "framework"]\nN_REPS = 20\n\n# Per-scenario means. The unweighted mean over S1-S4 reproduces the aggregate\n# values printed in the manuscript abstract, Section 3.1 and Table 4:\n#   detection latency 43.1 -> 8.5 s ; recovery time 399 -> 122 s\nTARGET_DETECTION = {\n    "baseline":  {"S1": 52.0, "S2": 68.4, "S3": 18.7, "S4": 33.3},   # mean 43.1\n    "framework": {"S1": 9.2,  "S2": 7.4,  "S3": 5.8,  "S4": 11.6},   # mean 8.5\n}\nTARGET_RECOVERY = {\n    "baseline":  {"S1": 402.0, "S2": 356.0, "S3": 511.0, "S4": 327.0},  # mean 399\n    "framework": {"S1": 118.0, "S2": 96.0,  "S3": 164.0, "S4": 110.0},  # mean 122\n}\n# Coefficient of variation of the lognormal run-to-run dispersion.\nCV_DETECTION = {"baseline": 0.32, "framework": 0.26}\nCV_RECOVERY = {"baseline": 0.24, "framework": 0.21}\n\n# NRI targets published for S3 only (Section 3.3 and Table 4).\nTARGET_NRI_S3 = {"baseline": 0.71, "framework": 0.93}\n\n# Availability-floor priors for the scenarios without a published NRI. They are\n# calibrated for S3 against TARGET_NRI_S3 and left at these values elsewhere.\nFLOOR_PRIOR = {\n    "baseline":  {"S1": 0.62, "S2": 0.78, "S3": 0.34, "S4": 0.70},\n    "framework": {"S1": 0.88, "S2": 0.94, "S3": 0.72, "S4": 0.91},\n}\n\nCFG = NRIConfig(rto=300.0, a_min=0.95, a_max=1.00, hold=30.0, sampling_interval=1.0)\nT_ATTACK = 120.0\nT_END = 1500.0\nNOISE_SD = 0.0025\n\n# Integrity verification: per-scenario operating point of the audit layer.\n# Observation unit = one challenged telemetry block.\nINTEGRITY_RATES = {\n    "S1": {"tpr": 0.972, "tnr": 0.9925, "n_pos": 900,  "n_neg": 3600},\n    "S2": {"tpr": 0.991, "tnr": 0.9890, "n_pos": 2400, "n_neg": 3600},\n    "S3": {"tpr": 0.958, "tnr": 0.9945, "n_pos": 600,  "n_neg": 3600},\n    "S4": {"tpr": 0.976, "tnr": 0.9935, "n_pos": 700,  "n_neg": 3600},\n}\nCORRUPTION_LEVELS = [0.01, 0.05, 0.10, 0.20]\n\n# Pooled integrity-verification accuracy printed in the abstract and Table 4.\nTARGET_INTEGRITY_ACCURACY = 0.987\n\n# Resource baselines (absolute units) and the framework\'s relative overhead.\nRESOURCE_SPEC = {\n    "cpu_pct":            {"baseline": 38.5,  "overhead": 0.054, "sd": 0.06},\n    "ram_mb":             {"baseline": 2410., "overhead": 0.041, "sd": 0.04},\n    "network_kbps":       {"baseline": 1875., "overhead": 0.032, "sd": 0.07},\n    "storage_mb_per_h":   {"baseline": 96.0,  "overhead": 0.058, "sd": 0.09},\n}\nLATENCY_SPEC = {\n    "integrity_verification_ms": (14.2, 0.35),\n    "graph_solver_ms":           (23.6, 0.30),\n    "whatif_simulation_ms":      (186.0, 0.28),\n    "end_to_end_orchestration_ms": (412.0, 0.25),\n}\n\n# Ablation / baseline matrix. Each variant declares which mechanisms it enables\n# and the operating point it produces, so the rates are stated rather than\n# derived from string matching on the variant name.\nABLATION = {\n    #                       det   rec   unsafe  viol   rollback success rank  twin_err\n    "B0_ids_manual":       dict(det=1.00, rec=1.00, unsafe=0.000, viol=0.000,\n                                rollback=0.000, success=0.84, rank=0.52, twin_err=None),\n    "B1_ids_playbook":     dict(det=1.00, rec=0.52, unsafe=0.115, viol=0.092,\n                                rollback=0.180, success=0.89, rank=0.52, twin_err=None),\n    "B2_stack_no_dt":      dict(det=0.42, rec=0.38, unsafe=0.072, viol=0.048,\n                                rollback=0.121, success=0.93, rank=0.61, twin_err=None),\n    "A1_no_graph":         dict(det=0.24, rec=0.34, unsafe=0.056, viol=0.021,\n                                rollback=0.094, success=0.94, rank=0.58, twin_err=0.083),\n    "A2_no_whatif":        dict(det=0.21, rec=0.31, unsafe=0.081, viol=0.037,\n                                rollback=0.142, success=0.93, rank=0.93, twin_err=None),\n    "FULL_framework":      dict(det=0.20, rec=0.31, unsafe=0.014, viol=0.000,\n                                rollback=0.038, success=0.98, rank=0.95, twin_err=0.041),\n}\n\n\n# --------------------------------------------------------------------------- #\n# helpers\n# --------------------------------------------------------------------------- #\ndef lognormal_with_mean(rng, mean: float, cv: float, n: int) -> np.ndarray:\n    """Draw n lognormal values, then rescale so the SAMPLE mean equals `mean`.\n\n    Rescaling is what makes the generated aggregates reproduce the published\n    means exactly; the shape of the dispersion is still lognormal.\n    """\n    sigma = np.sqrt(np.log(1.0 + cv ** 2))\n    mu = -0.5 * sigma ** 2\n    x = rng.lognormal(mu, sigma, size=n)\n    return x * (mean / x.mean())\n\n\ndef availability_trace(rng, t_attack: float, l_det: float, l_rec: float,\n                       floor: float, cfg: NRIConfig = CFG):\n    """Build one availability trace consistent with its detection and recovery times.\n\n    Phases: nominal -> exponential degradation from ``t_attack`` towards\n    ``floor`` -> containment at detection + reaction -> monotone recovery that\n    crosses ``a_min`` exactly at ``t_attack + l_rec`` and then settles at nominal.\n    Recovery measured back off this trace with Eq. (15) therefore equals ``l_rec``.\n    """\n    t = np.arange(0.0, T_END + cfg.sampling_interval, cfg.sampling_interval)\n    a = np.full(t.shape, 0.999)\n\n    t_contain = t_attack + l_det + 0.18 * l_rec\n    t_cross = t_attack + l_rec\n    if t_contain >= t_cross - 20.0:           # keep a physically ordered timeline\n        t_contain = t_cross - 20.0\n    tau_deg = max(6.0, 0.28 * (t_contain - t_attack))\n\n    deg = (t >= t_attack) & (t < t_contain)\n    a[deg] = floor + (0.999 - floor) * np.exp(-(t[deg] - t_attack) / tau_deg)\n    a_at_contain = floor + (0.999 - floor) * np.exp(-(t_contain - t_attack) / tau_deg)\n\n    # recovery: smoothstep from the containment level to 0.93 at t_cross - 1 s,\n    # then a fast ramp through a_min so the hold condition of Eq. (15) is met.\n    rec = (t >= t_contain) & (t < t_cross)\n    span = max(t_cross - t_contain, 1e-6)\n    u = (t[rec] - t_contain) / span\n    a[rec] = a_at_contain + (0.930 - a_at_contain) * (u ** 2 * (3 - 2 * u))\n\n    tail = t >= t_cross\n    u2 = np.clip((t[tail] - t_cross) / 25.0, 0.0, 1.0)\n    a[tail] = 0.962 + (0.999 - 0.962) * (u2 ** 2 * (3 - 2 * u2))\n\n    a = a + rng.normal(0.0, NOISE_SD, size=a.shape)\n    # Round here, not at write time: the metrics stored in run_level_metrics.csv\n    # must be computed from exactly the values the trace file contains, otherwise\n    # analysis/calculate_nri.py cannot reproduce them from the published traces.\n    return t, np.round(np.clip(a, 0.0, 1.0), 5)\n\n\ndef calibrate_floor(rng_seed: int, scenario: str, method: str, l_det, l_rec,\n                    target_nri: float, lo: float = 0.05, hi: float = 0.985):\n    """Bisect the availability floor so the mean NRI over the reps hits the target."""\n    def mean_nri(floor):\n        rng = np.random.default_rng(rng_seed)\n        vals = []\n        for k in range(len(l_det)):\n            t, a = availability_trace(rng, T_ATTACK, l_det[k], l_rec[k], floor)\n            below = np.flatnonzero(a < CFG.a_min)\n            t_dis = float(t[below[0]]) if below.size else T_ATTACK\n            vals.append(nri(t, a, t_dis, CFG))\n        return float(np.mean(vals))\n\n    for _ in range(60):\n        mid = 0.5 * (lo + hi)\n        if mean_nri(mid) < target_nri:\n            lo = mid\n        else:\n            hi = mid\n    return 0.5 * (lo + hi)\n\n\n# --------------------------------------------------------------------------- #\n# generators\n# --------------------------------------------------------------------------- #\n# Eq. (15) measures recovery from the first sample that begins a satisfied hold\n# window, which sits a fraction of a sampling interval later than the injected\n# availability crossing. RECOVERY_TARGET_OFFSET absorbs that systematic gap so\n# the MEASURED per-scenario mean matches the published value; it is calibrated\n# once (see PROVENANCE.md) and applied to the injected crossing time.\nRECOVERY_TARGET_OFFSET = -0.55\n\n\ndef generate_runs(out: Path):\n    master = np.random.default_rng(SEED)\n    rows, floors = [], {}\n    trace_dir = out / "availability_traces"\n    trace_dir.mkdir(parents=True, exist_ok=True)\n\n    for method in METHODS:\n        for scenario in SCENARIOS:\n            sub = np.random.default_rng(abs(hash((SEED, method, scenario))) % (2 ** 32))\n            l_det = lognormal_with_mean(sub, TARGET_DETECTION[method][scenario],\n                                        CV_DETECTION[method], N_REPS)\n            l_rec = lognormal_with_mean(\n                sub, TARGET_RECOVERY[method][scenario] + RECOVERY_TARGET_OFFSET,\n                CV_RECOVERY[method], N_REPS)\n            trace_seed = abs(hash((SEED, "trace", method, scenario))) % (2 ** 32)\n\n            if scenario == "S3":\n                floor = calibrate_floor(trace_seed, scenario, method, l_det, l_rec,\n                                        TARGET_NRI_S3[method])\n            else:\n                floor = FLOOR_PRIOR[method][scenario]\n            floors[(method, scenario)] = floor\n\n            rng = np.random.default_rng(trace_seed)\n            for k in range(N_REPS):\n                t, a = availability_trace(rng, T_ATTACK, l_det[k], l_rec[k], floor)\n                below = np.flatnonzero(a < CFG.a_min)\n                t_dis = float(t[below[0]]) if below.size else T_ATTACK\n                rec_measured = recovery_time(t, a, T_ATTACK, CFG)\n                nri_k = nri(t, a, t_dis, CFG)\n\n                run_id = f"{scenario}_{method}_r{k + 1:02d}"\n                pd.DataFrame({\n                    "t_s": t,\n                    "availability": a,\n                    "run_id": run_id,\n                    "data_origin": DATA_ORIGIN,\n                }).to_csv(trace_dir / f"{run_id}.csv", index=False)\n\n                rows.append({\n                    "run_id": run_id,\n                    "scenario": scenario,\n                    "method": method,\n                    "repetition": k + 1,\n                    "attack_onset_s": T_ATTACK,\n                    "detection_s": round(T_ATTACK + l_det[k], 3),\n                    "containment_s": round(T_ATTACK + l_det[k] + 0.18 * l_rec[k], 3),\n                    "recovery_s": round(T_ATTACK + rec_measured, 3),\n                    "disruption_onset_s": round(t_dis, 3),\n                    "detection_latency_s": round(l_det[k], 3),\n                    "recovery_time_s": round(rec_measured, 3),\n                    "availability_floor": round(floor, 5),\n                    "nri": round(nri_k, 5),\n                    "recovery_censored": int(np.isnan(rec_measured)),\n                    "trace_file": f"availability_traces/{run_id}.csv",\n                    "data_origin": DATA_ORIGIN,\n                })\n\n    df = pd.DataFrame(rows).sort_values(["scenario", "method", "repetition"])\n    df.to_csv(out / "run_level_metrics.csv", index=False)\n    return df, floors\n\n\ndef _confusion_cells():\n    """Per-scenario, per-corruption-level operating points before calibration."""\n    cells = []\n    for scenario in SCENARIOS:\n        spec = INTEGRITY_RATES[scenario]\n        for level in CORRUPTION_LEVELS:\n            # Sensitivity grows with the corrupted fraction: a larger corrupted\n            # share is hit by the same challenge budget with higher probability.\n            cells.append({\n                "scenario": scenario,\n                "corruption_fraction": level,\n                "tpr": min(0.9995, spec["tpr"] * (0.965 + 0.35 * level)),\n                "tnr": spec["tnr"],\n                "n_pos": int(round(spec["n_pos"] * (0.4 + 5.0 * level) / 4)),\n                "n_neg": int(round(spec["n_neg"] / len(CORRUPTION_LEVELS))),\n            })\n    return cells\n\n\ndef generate_confusion(out: Path, target_accuracy: float = TARGET_INTEGRITY_ACCURACY):\n    """Draw the confusion matrices, calibrated to the published pooled accuracy.\n\n    The per-cell error rates are scaled by one common factor so that the pooled\n    *expected* accuracy equals the published 98.7%; the realised counts are then\n    drawn binomially and reported with a Wilson interval rather than forced to\n    the target. The draw seed is fixed so the reference dataset reproduces the\n    published point value exactly.\n    """\n    # Offset 164 is the smallest offset in [1, 400) whose binomial draw reproduces\n    # the published pooled accuracy to four decimals; see PROVENANCE.md.\n    rng = np.random.default_rng(SEED + 164)\n    cdir = out / "confusion_matrices"\n    cdir.mkdir(parents=True, exist_ok=True)\n\n    cells = _confusion_cells()\n    n_total = sum(c["n_pos"] + c["n_neg"] for c in cells)\n    expected_errors = sum(c["n_pos"] * (1 - c["tpr"]) + c["n_neg"] * (1 - c["tnr"])\n                          for c in cells)\n    k = (1.0 - target_accuracy) * n_total / expected_errors\n\n    rows = []\n    for c in cells:\n        tpr = 1.0 - k * (1.0 - c["tpr"])\n        tnr = 1.0 - k * (1.0 - c["tnr"])\n        if not (0.0 < tpr < 1.0 and 0.0 < tnr < 1.0):\n            raise ValueError("calibration factor pushed a rate out of (0, 1)")\n        tp = int(rng.binomial(c["n_pos"], tpr))\n        fn = c["n_pos"] - tp\n        tn = int(rng.binomial(c["n_neg"], tnr))\n        fp = c["n_neg"] - tn\n        rows.append({"scenario": c["scenario"],\n                     "corruption_fraction": c["corruption_fraction"],\n                     "observation_unit": "challenged_telemetry_block",\n                     "tp": tp, "fn": fn, "tn": tn, "fp": fp,\n                     "n": tp + fn + tn + fp,\n                     "nominal_tpr": round(tpr, 5), "nominal_tnr": round(tnr, 5),\n                     "data_origin": DATA_ORIGIN})\n    df = pd.DataFrame(rows)\n    df.to_csv(cdir / "integrity_confusion.csv", index=False)\n    return df\n\n\ndef generate_resources(out: Path):\n    """Per-run resource and latency measurements.\n\n    Each metric is drawn per run and then rescaled so that the *arm* mean equals\n    its intended value exactly. Without that step the sampling error of 80 runs\n    moves the realised relative overhead of Eq. (17) by up to two percentage\n    points, which would silently contradict the published "below 6%" bound.\n    """\n    rng = np.random.default_rng(SEED + 2)\n    rdir = out / "resource_measurements"\n    rdir.mkdir(parents=True, exist_ok=True)\n    n_arm = len(SCENARIOS) * N_REPS\n    index = [(s, k + 1) for s in SCENARIOS for k in range(N_REPS)]\n\n    frames = {}\n    for method in METHODS:\n        cols = {}\n        for metric, spec in RESOURCE_SPEC.items():\n            target = spec["baseline"] * (1 + spec["overhead"]) if method == "framework" \\\n                else spec["baseline"]\n            x = 1.0 + rng.normal(0.0, spec["sd"], size=n_arm)\n            cols[metric] = x * (target / x.mean())\n        for metric, (mean, cv) in LATENCY_SPEC.items():\n            if method == "baseline" and metric != "end_to_end_orchestration_ms":\n                cols[metric] = np.full(n_arm, np.nan)  # component absent in the baseline\n                continue\n            target = mean if method == "framework" else mean * 7.4\n            cols[metric] = lognormal_with_mean(rng, target, cv, n_arm)\n        frames[method] = cols\n\n    rows = []\n    for method in METHODS:\n        for i, (scenario, rep) in enumerate(index):\n            row = {"run_id": f"{scenario}_{method}_r{rep:02d}", "scenario": scenario,\n                   "method": method, "repetition": rep}\n            for metric, values in frames[method].items():\n                v = values[i]\n                row[metric] = float("nan") if np.isnan(v) else round(float(v), 3)\n            row["data_origin"] = DATA_ORIGIN\n            rows.append(row)\n    df = pd.DataFrame(rows)\n    df.to_csv(rdir / "resource_usage.csv", index=False)\n    return df\n\n\ndef generate_ablation(out: Path):\n    """Baseline and ablation variants of the experimental matrix.\n\n    ``twin_err = None`` marks a variant that runs no digital twin at all; the\n    prediction-error column is left empty for those rows instead of being filled\n    with a value that has no referent.\n    """\n    rng = np.random.default_rng(SEED + 3)\n    rows = []\n    for variant, spec in ABLATION.items():\n        for scenario in SCENARIOS:\n            det = lognormal_with_mean(\n                rng, TARGET_DETECTION["baseline"][scenario] * spec["det"], 0.30, N_REPS)\n            rec = lognormal_with_mean(\n                rng, TARGET_RECOVERY["baseline"][scenario] * spec["rec"], 0.23, N_REPS)\n            fast = spec["det"] < 0.5           # variants that decide on the twin\n            for k in range(N_REPS):\n                rows.append({\n                    "variant": variant, "scenario": scenario, "repetition": k + 1,\n                    "detection_latency_s": round(float(det[k]), 3),\n                    "recovery_time_s": round(float(rec[k]), 3),\n                    "unsafe_action": int(rng.random() < spec["unsafe"]),\n                    "policy_violation": int(rng.random() < spec["viol"]),\n                    "rollback": int(rng.random() < spec["rollback"]),\n                    "recovery_success": int(rng.random() < spec["success"]),\n                    "orchestration_decision_latency_ms": round(\n                        float(412.0 * (0.35 if fast else 1.9) * rng.lognormal(0, 0.22)), 3),\n                    "twin_prediction_error": (\n                        round(float(abs(rng.normal(spec["twin_err"], 0.02))), 4)\n                        if spec["twin_err"] is not None else float("nan")),\n                    "risk_ranking_correct": int(rng.random() < spec["rank"]),\n                    "data_origin": DATA_ORIGIN,\n                })\n    df = pd.DataFrame(rows)\n    df.to_csv(out / "ablation_runs.csv", index=False)\n    return df\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=__doc__,\n                                 formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--out", default="data", help="output data directory")\n    args = ap.parse_args()\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n\n    runs, floors = generate_runs(out)\n    conf = generate_confusion(out)\n    res = generate_resources(out)\n    abl = generate_ablation(out)\n\n    det = runs.groupby("method")["detection_latency_s"].mean()\n    rec = runs.groupby("method")["recovery_time_s"].mean()\n    scen_det = runs.groupby(["method", "scenario"])["detection_latency_s"].mean()\n    scen_rec = runs.groupby(["method", "scenario"])["recovery_time_s"].mean()\n    nri_s3 = runs[runs.scenario == "S3"].groupby("method")["nri"].mean()\n    acc = (conf.tp.sum() + conf.tn.sum()) / conf.n.sum()\n    fpr = conf.fp.sum() / (conf.fp.sum() + conf.tn.sum())\n\n    manifest = {\n        "generator": "simulate_reference_dataset.py",\n        "seed": SEED,\n        "data_origin": DATA_ORIGIN,\n        "n_runs": int(len(runs)),\n        "n_traces": int(len(runs)),\n        "reproduced_aggregates": {\n            "detection_latency_mean_s": {m: round(float(det[m]), 4) for m in METHODS},\n            "recovery_time_mean_s": {m: round(float(rec[m]), 4) for m in METHODS},\n            "scenario_mean_detection_s": {f"{m}/{s}": round(float(scen_det[(m, s)]), 3)\n                                          for m in METHODS for s in SCENARIOS},\n            "scenario_mean_recovery_s": {f"{m}/{s}": round(float(scen_rec[(m, s)]), 3)\n                                         for m in METHODS for s in SCENARIOS},\n            "nri_s3_mean": {m: round(float(nri_s3[m]), 4) for m in METHODS},\n            "integrity_accuracy": round(float(acc), 5),\n            "integrity_fpr": round(float(fpr), 5),\n        },\n        "availability_floor_calibrated": {f"{m}/{s}": round(v, 5)\n                                          for (m, s), v in floors.items()},\n        "warning": ("Synthetic reference data. Not a measurement. Replace every file "\n                    "in data/ with real measurement exports before submission."),\n    }\n    (out / "generation_manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n\n    print(json.dumps(manifest["reproduced_aggregates"], indent=2))\n    print(f"\\nwrote {len(runs)} runs, {len(conf)} confusion cells, "\n          f"{len(res)} resource rows, {len(abl)} ablation rows -> {out}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
    SCRIPTS['statistics.py'] = '#!/usr/bin/env python3\n"""Compute every statistic reported in the revised Results section.\n\nReads ``data/`` and writes machine-readable tables to ``results/``:\n\n  table_S1_detection_latency.csv   scenario-level detection latency\n  table_S2_recovery_time.csv       scenario-level recovery time\n  table_S3_nri.csv                 scenario-level NRI and resilience deficit\n  table_S4_integrity.csv           confusion matrices with Wilson intervals\n  table_S5_overhead.csv            resource overhead per Eq. (17), both denominators\n  table_S6_ablation.csv            ablation variants and safety-of-action rates\n  summary.json                     the aggregate values quoted in the abstract\n\nUsage:  python analysis/statistics.py --data data --out results\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom dtcr import stats as st  # noqa: E402\n\nSCENARIOS = ["S1", "S2", "S3", "S4"]\nPAIRED = True  # runs were scheduled as interleaved matched pairs; see PROTOCOL.md\n\n\ndef _row(scenario, metric, comp):\n    b, f = comp["baseline"], comp["framework"]\n    return {\n        "scenario": scenario, "metric": metric,\n        "n_baseline": b["n"], "n_framework": f["n"],\n        "baseline_mean": b["mean"], "baseline_sd": b["sd"],\n        "baseline_median": b["median"], "baseline_iqr": b["iqr"],\n        "baseline_ci95_lo": b["ci95_lo"], "baseline_ci95_hi": b["ci95_hi"],\n        "framework_mean": f["mean"], "framework_sd": f["sd"],\n        "framework_median": f["median"], "framework_iqr": f["iqr"],\n        "framework_ci95_lo": f["ci95_lo"], "framework_ci95_hi": f["ci95_hi"],\n        "abs_diff_mean": comp["abs_diff_mean"],\n        "diff_ci95_lo": comp.get("diff_ci", {}).get("ci95_lo"),\n        "diff_ci95_hi": comp.get("diff_ci", {}).get("ci95_hi"),\n        "rel_reduction_pct": comp["rel_reduction_pct"],\n        "hedges_g": comp["hedges_g"], "cliffs_delta": comp["cliffs_delta"],\n        "test": comp["test"]["name"], "p_value": comp["test"]["p_value"],\n        "test_nonparametric": comp["test_nonparametric"]["name"],\n        "p_value_nonparametric": comp["test_nonparametric"]["p_value"],\n        "paired": comp["paired"],\n    }\n\n\ndef metric_table(runs: pd.DataFrame, column: str, metric_name: str) -> pd.DataFrame:\n    rows, pvals = [], []\n    for s in SCENARIOS:\n        sub = runs[runs.scenario == s]\n        b = sub[sub.method == "baseline"].sort_values("repetition")[column].to_numpy()\n        f = sub[sub.method == "framework"].sort_values("repetition")[column].to_numpy()\n        comp = st.compare_groups(b, f, paired=PAIRED)\n        rows.append(_row(s, metric_name, comp))\n        pvals.append(comp["test"]["p_value"])\n    b_all = runs[runs.method == "baseline"][column].to_numpy()\n    f_all = runs[runs.method == "framework"][column].to_numpy()\n    rows.append(_row("pooled", metric_name,\n                     st.compare_groups(b_all, f_all, paired=PAIRED)))\n    holm = st.holm_bonferroni(pvals)\n    df = pd.DataFrame(rows)\n    df["p_value_holm"] = holm["p_adjusted"] + [float("nan")]\n    return df\n\n\ndef nri_table(runs: pd.DataFrame) -> pd.DataFrame:\n    rows, pvals = [], []\n    for s in SCENARIOS + ["pooled"]:\n        sub = runs if s == "pooled" else runs[runs.scenario == s]\n        b = sub[sub.method == "baseline"].sort_values(["scenario", "repetition"])["nri"].to_numpy()\n        f = sub[sub.method == "framework"].sort_values(["scenario", "repetition"])["nri"].to_numpy()\n        comp = st.compare_groups(b, f, paired=PAIRED)\n        # NRI is higher-is-better: report the gain and the deficit reduction.\n        bm, fm = comp["baseline"]["mean"], comp["framework"]["mean"]\n        rows.append({\n            "scenario": s,\n            "n": comp["baseline"]["n"],\n            "baseline_mean": bm, "baseline_sd": comp["baseline"]["sd"],\n            "baseline_ci95_lo": comp["baseline"]["ci95_lo"],\n            "baseline_ci95_hi": comp["baseline"]["ci95_hi"],\n            "framework_mean": fm, "framework_sd": comp["framework"]["sd"],\n            "framework_ci95_lo": comp["framework"]["ci95_lo"],\n            "framework_ci95_hi": comp["framework"]["ci95_hi"],\n            "absolute_gain": fm - bm,\n            "relative_gain_pct": 100.0 * (fm - bm) / bm if bm else float("nan"),\n            "deficit_baseline": 1 - bm, "deficit_framework": 1 - fm,\n            "deficit_reduction_pct": (100.0 * ((1 - bm) - (1 - fm)) / (1 - bm)\n                                      if (1 - bm) else float("nan")),\n            "hedges_g": comp["hedges_g"], "cliffs_delta": comp["cliffs_delta"],\n            "test": comp["test"]["name"], "p_value": comp["test"]["p_value"],\n        })\n        if s != "pooled":\n            pvals.append(comp["test"]["p_value"])\n    df = pd.DataFrame(rows)\n    df["p_value_holm"] = st.holm_bonferroni(pvals)["p_adjusted"] + [float("nan")]\n    return df\n\n\ndef integrity_table(conf: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    for _, r in conf.iterrows():\n        m = st.classification_metrics(int(r.tp), int(r.tn), int(r.fp), int(r.fn))\n        rows.append({"scenario": r.scenario, "corruption_fraction": r.corruption_fraction,\n                     "observation_unit": r.observation_unit, **_flatten(m)})\n    for s in SCENARIOS:\n        sub = conf[conf.scenario == s]\n        m = st.classification_metrics(int(sub.tp.sum()), int(sub.tn.sum()),\n                                      int(sub.fp.sum()), int(sub.fn.sum()))\n        rows.append({"scenario": s, "corruption_fraction": "all",\n                     "observation_unit": "challenged_telemetry_block", **_flatten(m)})\n    m = st.classification_metrics(int(conf.tp.sum()), int(conf.tn.sum()),\n                                  int(conf.fp.sum()), int(conf.fn.sum()))\n    rows.append({"scenario": "pooled", "corruption_fraction": "all",\n                 "observation_unit": "challenged_telemetry_block", **_flatten(m)})\n    return pd.DataFrame(rows)\n\n\ndef _flatten(m: dict) -> dict:\n    out = {}\n    for k, v in m.items():\n        if isinstance(v, tuple):\n            out[f"{k}_lo"], out[f"{k}_hi"] = v\n        else:\n            out[k] = v\n    return out\n\n\ndef overhead_table(res: pd.DataFrame, cluster_capacity: dict) -> pd.DataFrame:\n    """Eq. (17) overhead plus the share-of-capacity denominator.\n\n    The manuscript uses Eq. (17) (relative to baseline consumption) in Section 2.8\n    and \'share of cluster capacity\' in Section 3.2. Both are reported here so the\n    denominator of every overhead number is unambiguous.\n    """\n    rows = []\n    metrics = [c for c in res.columns\n               if c not in {"run_id", "scenario", "method", "repetition", "data_origin"}]\n    for metric in metrics:\n        b = res[res.method == "baseline"][metric].to_numpy(float)\n        f = res[res.method == "framework"][metric].to_numpy(float)\n        b, f = b[~np.isnan(b)], f[~np.isnan(f)]\n        if b.size == 0 or f.size == 0:\n            rows.append({"metric": metric, "baseline_mean": float("nan"),\n                         "framework_mean": float(f.mean()) if f.size else float("nan"),\n                         "note": "component absent in the baseline arm"})\n            continue\n        db, df_ = st.describe(b), st.describe(f)\n        rel = 100.0 * (df_["mean"] - db["mean"]) / db["mean"]\n        boot = st.bootstrap_ci(f - b[:f.size]) if b.size == f.size else None\n        cap = cluster_capacity.get(metric)\n        rows.append({\n            "metric": metric,\n            "baseline_mean": db["mean"], "baseline_sd": db["sd"],\n            "baseline_p95": float(np.percentile(b, 95)),\n            "framework_mean": df_["mean"], "framework_sd": df_["sd"],\n            "framework_p95": float(np.percentile(f, 95)),\n            "absolute_difference": df_["mean"] - db["mean"],\n            "relative_overhead_pct_eq17": rel,\n            "diff_ci95_lo": boot["ci95_lo"] if boot else None,\n            "diff_ci95_hi": boot["ci95_hi"] if boot else None,\n            "cluster_capacity": cap,\n            "share_of_capacity_pct": (100.0 * (df_["mean"] - db["mean"]) / cap\n                                      if cap else None),\n        })\n    return pd.DataFrame(rows)\n\n\ndef ablation_table(abl: pd.DataFrame) -> pd.DataFrame:\n    g = abl.groupby("variant")\n    df = pd.DataFrame({\n        "detection_latency_mean_s": g["detection_latency_s"].mean(),\n        "detection_latency_sd_s": g["detection_latency_s"].std(ddof=1),\n        "recovery_time_mean_s": g["recovery_time_s"].mean(),\n        "recovery_time_sd_s": g["recovery_time_s"].std(ddof=1),\n        "unsafe_action_rate": g["unsafe_action"].mean(),\n        "policy_violation_rate": g["policy_violation"].mean(),\n        "rollback_rate": g["rollback"].mean(),\n        "recovery_success_rate": g["recovery_success"].mean(),\n        "orchestration_decision_latency_mean_ms": g["orchestration_decision_latency_ms"].mean(),\n        "twin_prediction_error_mean": g["twin_prediction_error"].mean(),\n        "risk_ranking_accuracy": g["risk_ranking_correct"].mean(),\n        "n": g.size(),\n    }).reset_index()\n    for col, num in [("unsafe_action_rate", "unsafe_action"),\n                     ("policy_violation_rate", "policy_violation"),\n                     ("rollback_rate", "rollback"),\n                     ("recovery_success_rate", "recovery_success"),\n                     ("risk_ranking_accuracy", "risk_ranking_correct")]:\n        lo, hi = [], []\n        for v in df.variant:\n            sub = abl[abl.variant == v]\n            a, b = st.wilson_ci(int(sub[num].sum()), len(sub))\n            lo.append(a); hi.append(b)\n        df[f"{col}_ci95_lo"], df[f"{col}_ci95_hi"] = lo, hi\n    return df\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=__doc__,\n                                 formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--out", default="results")\n    args = ap.parse_args()\n    data, out = Path(args.data), Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n\n    runs = pd.read_csv(data / "run_level_metrics.csv")\n    conf = pd.read_csv(data / "confusion_matrices" / "integrity_confusion.csv")\n    res = pd.read_csv(data / "resource_measurements" / "resource_usage.csv")\n    abl = pd.read_csv(data / "ablation_runs.csv")\n\n    # Cluster capacity of the testbed described in Table 2 (4 edge nodes + 3 VMs).\n    capacity = {"cpu_pct": 700.0, "ram_mb": 80384.0,\n                "network_kbps": 1000000.0, "storage_mb_per_h": 4096.0}\n\n    t_det = metric_table(runs, "detection_latency_s", "detection_latency_s")\n    t_rec = metric_table(runs, "recovery_time_s", "recovery_time_s")\n    t_nri = nri_table(runs)\n    t_int = integrity_table(conf)\n    t_ovh = overhead_table(res, capacity)\n    t_abl = ablation_table(abl)\n\n    t_det.to_csv(out / "table_S1_detection_latency.csv", index=False)\n    t_rec.to_csv(out / "table_S2_recovery_time.csv", index=False)\n    t_nri.to_csv(out / "table_S3_nri.csv", index=False)\n    t_int.to_csv(out / "table_S4_integrity.csv", index=False)\n    t_ovh.to_csv(out / "table_S5_overhead.csv", index=False)\n    t_abl.to_csv(out / "table_S6_ablation.csv", index=False)\n\n    pooled_det = t_det[t_det.scenario == "pooled"].iloc[0]\n    pooled_rec = t_rec[t_rec.scenario == "pooled"].iloc[0]\n    s3 = t_nri[t_nri.scenario == "S3"].iloc[0]\n    pooled_int = t_int[t_int.scenario == "pooled"].iloc[0]\n    summary = {\n        "data_origin": str(runs.data_origin.iloc[0]),\n        "n_per_cell": int(pooled_det.n_baseline / len(SCENARIOS)),\n        "detection_latency_s": {\n            "baseline_mean": float(pooled_det.baseline_mean),\n            "baseline_ci95": [float(pooled_det.baseline_ci95_lo),\n                              float(pooled_det.baseline_ci95_hi)],\n            "framework_mean": float(pooled_det.framework_mean),\n            "framework_ci95": [float(pooled_det.framework_ci95_lo),\n                               float(pooled_det.framework_ci95_hi)],\n            "relative_reduction_pct": float(pooled_det.rel_reduction_pct),\n            "hedges_g": float(pooled_det.hedges_g),\n            "p_value": float(pooled_det.p_value)},\n        "recovery_time_s": {\n            "baseline_mean": float(pooled_rec.baseline_mean),\n            "baseline_ci95": [float(pooled_rec.baseline_ci95_lo),\n                              float(pooled_rec.baseline_ci95_hi)],\n            "framework_mean": float(pooled_rec.framework_mean),\n            "framework_ci95": [float(pooled_rec.framework_ci95_lo),\n                               float(pooled_rec.framework_ci95_hi)],\n            "relative_reduction_pct": float(pooled_rec.rel_reduction_pct),\n            "hedges_g": float(pooled_rec.hedges_g),\n            "p_value": float(pooled_rec.p_value)},\n        "nri_S3": {\n            "baseline_mean": float(s3.baseline_mean),\n            "baseline_ci95": [float(s3.baseline_ci95_lo), float(s3.baseline_ci95_hi)],\n            "framework_mean": float(s3.framework_mean),\n            "framework_ci95": [float(s3.framework_ci95_lo), float(s3.framework_ci95_hi)],\n            "relative_gain_pct": float(s3.relative_gain_pct),\n            "deficit_reduction_pct": float(s3.deficit_reduction_pct)},\n        "integrity_pooled": {\n            "n": int(pooled_int.n), "accuracy": float(pooled_int.accuracy),\n            "accuracy_ci95": [float(pooled_int.accuracy_ci95_lo),\n                              float(pooled_int.accuracy_ci95_hi)],\n            "recall": float(pooled_int.recall), "precision": float(pooled_int.precision),\n            "specificity": float(pooled_int.specificity),\n            "f1": float(pooled_int.f1), "mcc": float(pooled_int.mcc),\n            "fpr": float(pooled_int.fpr),\n            "fpr_ci95": [float(pooled_int.fpr_ci95_lo), float(pooled_int.fpr_ci95_hi)]},\n        "overhead_max_relative_pct_eq17": float(\n            t_ovh.relative_overhead_pct_eq17.max(skipna=True)),\n        "overhead_max_share_of_capacity_pct": float(\n            t_ovh.share_of_capacity_pct.max(skipna=True)),\n    }\n    (out / "summary.json").write_text(json.dumps(summary, indent=2) + "\\n")\n    print(json.dumps(summary, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
    SCRIPTS['calculate_nri.py'] = '#!/usr/bin/env python3\n"""Recompute the normalized resilience index directly from the availability traces.\n\nThis script is the answer to the reviewer objection that Figure 6 could not be\nreconciled with Figure 5 or with the published NRI values.  It:\n\n  1. reads every availability trace in ``data/availability_traces/``;\n  2. recomputes the recovery time (Eq. 15) and the NRI (Eq. 18-19) with the\n     window parameters stated in ``configs/framework_parameters.yaml``;\n  3. cross-checks the recomputed values against ``data/run_level_metrics.csv``\n     and fails loudly on any mismatch;\n  4. writes per-run NRI values and the mean availability trajectory with a 95%\n     confidence band, which is what Figure 6 plots.\n\nEvery quantity the index depends on - RTO, A_min, A_max, the hold interval, the\nsampling interval, and the start of the integration window - is read from the\nconfiguration file and echoed into ``results/nri_parameters.json`` so the figure\nis reproducible from published inputs alone.\n\nUsage:  python analysis/calculate_nri.py --data data --out results\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport yaml\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom dtcr.resilience import NRIConfig, nri, recovery_time  # noqa: E402\nfrom dtcr import stats as st  # noqa: E402\n\n# run_level_metrics.csv stores the NRI rounded to five decimals, so the stored\n# and recomputed values may differ by up to one unit in the last stored place.\n# Anything larger means the traces and the summary file no longer describe the\n# same experiment.\nTOL_NRI = 1e-5   # one unit in the last stored place\nTOL_RECOVERY = 1e-9\n\n\ndef load_config(path: Path) -> NRIConfig:\n    cfg = yaml.safe_load(path.read_text())["resilience"]\n    return NRIConfig(rto=float(cfg["rto_s"]), a_min=float(cfg["a_min"]),\n                     a_max=float(cfg["a_max"]), hold=float(cfg["hold_interval_s"]),\n                     sampling_interval=float(cfg["sampling_interval_s"]))\n\n\ndef recompute(data: Path, cfg: NRIConfig) -> pd.DataFrame:\n    runs = pd.read_csv(data / "run_level_metrics.csv")\n    out = []\n    for _, r in runs.iterrows():\n        tr = pd.read_csv(data / r.trace_file)\n        t = tr.t_s.to_numpy(float)\n        a = tr.availability.to_numpy(float)\n        below = np.flatnonzero(a < cfg.a_min)\n        t_dis = float(t[below[0]]) if below.size else float(r.attack_onset_s)\n        out.append({\n            "run_id": r.run_id, "scenario": r.scenario, "method": r.method,\n            "repetition": int(r.repetition),\n            "t_dis_s": t_dis,\n            "window_start_s": t_dis, "window_end_s": t_dis + 2 * cfg.rto,\n            "recovery_time_s": recovery_time(t, a, float(r.attack_onset_s), cfg),\n            "nri": nri(t, a, t_dis, cfg),\n            "stored_recovery_time_s": float(r.recovery_time_s),\n            "stored_nri": float(r.nri),\n        })\n    df = pd.DataFrame(out)\n    df["delta_nri"] = df.nri - df.stored_nri\n    df["delta_recovery_s"] = df.recovery_time_s - df.stored_recovery_time_s\n    return df\n\n\ndef mean_trajectory(data: Path, runs: pd.DataFrame, scenario: str, method: str,\n                    cfg: NRIConfig) -> pd.DataFrame:\n    """Mean availability trajectory with a 95% t confidence band, aligned on t_dis."""\n    sub = runs[(runs.scenario == scenario) & (runs.method == method)]\n    grid = np.arange(-60.0, 2 * cfg.rto + 1.0, cfg.sampling_interval)\n    curves = []\n    for _, r in sub.iterrows():\n        tr = pd.read_csv(data / f"availability_traces/{r.run_id}.csv")\n        t = tr.t_s.to_numpy(float)\n        a = tr.availability.to_numpy(float)\n        below = np.flatnonzero(a < cfg.a_min)\n        t_dis = float(t[below[0]]) if below.size else 0.0\n        curves.append(np.interp(grid, t - t_dis, a))\n    m = np.vstack(curves)\n    mean = m.mean(axis=0)\n    sd = m.std(axis=0, ddof=1)\n    n = m.shape[0]\n    from scipy import stats as sps\n    half = sps.t.ppf(0.975, n - 1) * sd / np.sqrt(n)\n    return pd.DataFrame({"t_rel_s": grid, "mean": mean, "sd": sd,\n                         "ci95_lo": np.clip(mean - half, 0, 1),\n                         "ci95_hi": np.clip(mean + half, 0, 1),\n                         "n": n, "scenario": scenario, "method": method})\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=__doc__,\n                                 formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--configs", default="configs/framework_parameters.yaml")\n    ap.add_argument("--out", default="results")\n    ap.add_argument("--strict", action="store_true",\n                    help="exit non-zero if a recomputed value disagrees with the stored one")\n    args = ap.parse_args()\n    data, out = Path(args.data), Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n\n    cfg = load_config(Path(args.configs))\n    df = recompute(data, cfg)\n    df.to_csv(out / "nri_per_run.csv", index=False)\n\n    bad = df[(df.delta_nri.abs() > TOL_NRI) |\n             (df.delta_recovery_s.abs() > TOL_RECOVERY)]\n    traj = pd.concat([mean_trajectory(data, df, s, m, cfg)\n                      for s in sorted(df.scenario.unique())\n                      for m in ("baseline", "framework")], ignore_index=True)\n    traj.to_csv(out / "availability_trajectories.csv", index=False)\n\n    summary = {}\n    for s in sorted(df.scenario.unique()):\n        entry = {}\n        for m in ("baseline", "framework"):\n            v = df[(df.scenario == s) & (df.method == m)].nri.to_numpy()\n            d = st.describe(v)\n            entry[m] = {"n": d["n"], "mean": d["mean"], "sd": d["sd"],\n                        "ci95": [d["ci95_lo"], d["ci95_hi"]],\n                        "median": d["median"], "iqr": d["iqr"]}\n        bm, fm = entry["baseline"]["mean"], entry["framework"]["mean"]\n        entry["absolute_gain"] = fm - bm\n        entry["relative_gain_pct"] = 100 * (fm - bm) / bm\n        entry["deficit_reduction_pct"] = 100 * ((1 - bm) - (1 - fm)) / (1 - bm)\n        summary[s] = entry\n\n    params = {\n        "rto_s": cfg.rto, "a_min": cfg.a_min, "a_max": cfg.a_max,\n        "hold_interval_s": cfg.hold, "sampling_interval_s": cfg.sampling_interval,\n        "integration_window": "[t_dis, t_dis + 2*RTO]",\n        "t_dis_definition": "first sample with availability < a_min after attack onset",\n        "note": ("t_det (detection) and t_dis (disruption) are distinct symbols; "\n                 "the NRI window is anchored on t_dis, not on t_det."),\n        "n_traces": int(len(df)),\n        "consistency_check": {"max_abs_delta_nri": float(df.delta_nri.abs().max()),\n                              "max_abs_delta_recovery_s": float(df.delta_recovery_s.abs().max()),\n                              "mismatched_runs": int(len(bad))},\n        "per_scenario": summary,\n    }\n    (out / "nri_parameters.json").write_text(json.dumps(params, indent=2) + "\\n")\n    print(json.dumps(params, indent=2))\n\n    if len(bad):\n        print(f"\\nWARNING: {len(bad)} run(s) disagree with run_level_metrics.csv",\n              file=sys.stderr)\n        print(bad[["run_id", "delta_nri", "delta_recovery_s"]].to_string(index=False),\n              file=sys.stderr)\n        if args.strict:\n            return 1\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
    SCRIPTS['generate_figures.py'] = '#!/usr/bin/env python3\n"""Regenerate every manuscript figure from the published data.\n\nFigures written to ``figures/`` (PNG at 600 dpi and PDF):\n\n  figure5   detection latency and recovery time per scenario, with individual\n            run points, box plots and mean +/- 95% CI (replaces the bar chart)\n  figure6   mean availability trajectory with a 95% confidence band, the NRI\n            integration window, and the per-run NRI distribution\n  figure7   analytical audit-detection sensitivity (exact and lower bound)\n  figure8   dependency-risk propagation example and lambda sensitivity\n  figure9   integrity confusion metrics by corruption level (new)\n  figure10  resource overhead with both denominators (new)\n  figure11  ablation study across framework variants (new)\n\nUsage:  python analysis/generate_figures.py --data data --results results --out figures\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt          # noqa: E402\nimport numpy as np                        # noqa: E402\nimport pandas as pd                       # noqa: E402\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom dtcr import audit, risk              # noqa: E402\nfrom dtcr.resilience import NRIConfig     # noqa: E402\nfrom calculate_nri import load_config     # noqa: E402\n\nSCENARIOS = ["S1", "S2", "S3", "S4"]\nC_BASE, C_FRAME = "#B4442E", "#2F6F8F"\nC_GRID = "#D8D8D8"\n\nplt.rcParams.update({\n    "font.family": "DejaVu Sans", "font.size": 9,\n    "axes.grid": True, "grid.color": C_GRID, "grid.linewidth": 0.6,\n    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,\n    "figure.dpi": 120, "savefig.bbox": "tight",\n})\n\n\ndef save(fig, out: Path, name: str):\n    out.mkdir(parents=True, exist_ok=True)\n    fig.savefig(out / f"{name}.png", dpi=600)\n    fig.savefig(out / f"{name}.pdf")\n    plt.close(fig)\n    print(f"  wrote {name}.png / {name}.pdf")\n\n\ndef _paired_panel(ax, runs, column, ylabel, title):\n    """Individual points + box + mean with 95% CI, per scenario and arm."""\n    width, offset = 0.30, 0.19\n    for i, s in enumerate(SCENARIOS):\n        for j, (method, colour) in enumerate([("baseline", C_BASE),\n                                              ("framework", C_FRAME)]):\n            v = runs[(runs.scenario == s) & (runs.method == method)][column].to_numpy(float)\n            v = v[~np.isnan(v)]\n            x = i + (-offset if j == 0 else offset)\n            bp = ax.boxplot(v, positions=[x], widths=width, showfliers=False,\n                            patch_artist=True, medianprops=dict(color="#222", lw=1.2),\n                            whiskerprops=dict(color=colour, lw=1.0),\n                            capprops=dict(color=colour, lw=1.0))\n            bp["boxes"][0].set(facecolor=colour, alpha=0.20, edgecolor=colour, lw=1.1)\n            jitter = (np.random.default_rng(hash((s, method)) % 2**32).uniform(-0.075, 0.075, v.size))\n            ax.plot(x + jitter, v, "o", ms=2.6, color=colour, alpha=0.65, mec="none", zorder=3)\n            m = v.mean()\n            se = v.std(ddof=1) / np.sqrt(v.size)\n            from scipy import stats as sps\n            h = sps.t.ppf(0.975, v.size - 1) * se\n            ax.errorbar(x, m, yerr=h, fmt="D", ms=4.2, color=colour, mec="white",\n                        mew=0.7, ecolor=colour, elinewidth=1.6, capsize=3.5, zorder=4)\n    ax.set_xticks(range(len(SCENARIOS)))\n    ax.set_xticklabels(SCENARIOS)\n    ax.set_xlabel("Scenario")\n    ax.set_ylabel(ylabel)\n    ax.set_title(title, loc="left", fontsize=9.5, fontweight="bold")\n    ax.set_xlim(-0.6, len(SCENARIOS) - 0.4)\n\n\ndef figure5(runs: pd.DataFrame, out: Path):\n    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.7))\n    _paired_panel(axes[0], runs, "detection_latency_s",\n                  "Detection latency (s)", "(a) Detection latency")\n    _paired_panel(axes[1], runs, "recovery_time_s",\n                  "Service recovery time (s)", "(b) Service recovery time")\n    handles = [plt.Line2D([], [], color=C_BASE, marker="D", ls="", ms=5,\n                          label="IDS-only + manual recovery (baseline)"),\n               plt.Line2D([], [], color=C_FRAME, marker="D", ls="", ms=5,\n                          label="Proposed framework")]\n    fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False,\n               bbox_to_anchor=(0.5, -0.09))\n    fig.text(0.5, -0.17, "n = 20 paired repetitions per scenario and arm; boxes show the "\n             "median and IQR, diamonds the mean with a 95% confidence interval, "\n             "dots the individual runs.", ha="center", fontsize=7.4, color="#555")\n    save(fig, out, "figure5_detection_recovery")\n\n\ndef figure6(traj: pd.DataFrame, per_run: pd.DataFrame, cfg: NRIConfig, out: Path,\n            scenario: str = "S3"):\n    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.7),\n                             gridspec_kw={"width_ratios": [2.1, 1]})\n    ax = axes[0]\n    for method, colour, label in [("baseline", C_BASE, "Baseline"),\n                                  ("framework", C_FRAME, "Proposed framework")]:\n        t = traj[(traj.scenario == scenario) & (traj.method == method)]\n        ax.fill_between(t.t_rel_s, t.ci95_lo, t.ci95_hi, color=colour, alpha=0.20, lw=0)\n        ax.plot(t.t_rel_s, t["mean"], color=colour, lw=1.9, label=label)\n    ax.axhline(cfg.a_min, color="#444", ls="--", lw=1.0)\n    ax.annotate(f"$A_{{min}}$ = {cfg.a_min:g}", xy=(-58, cfg.a_min),\n                xytext=(0, -12), textcoords="offset points", fontsize=7.6,\n                color="#444", ha="left")\n    ax.axvspan(0, 2 * cfg.rto, color="#999", alpha=0.07, lw=0)\n    ax.axvline(0, color="#666", lw=0.9)\n    ax.axvline(2 * cfg.rto, color="#666", lw=0.9)\n    ax.annotate("NRI integration window\\n$[t_{dis},\\; t_{dis}+2\\\\,RTO]$",\n                xy=(cfg.rto, 0.30), ha="center", fontsize=7.6, color="#444")\n    ax.set_xlabel("Time relative to disruption onset $t_{dis}$ (s)")\n    ax.set_ylabel("Service availability $A(t)$")\n    ax.set_title(f"(a) Availability trajectory, {scenario}", loc="left",\n                 fontsize=9.5, fontweight="bold")\n    ax.set_ylim(0.15, 1.02)\n    ax.set_xlim(-60, 2 * cfg.rto)\n    ax.legend(frameon=False, loc="lower right", fontsize=8)\n\n    ax2 = axes[1]\n    data, colours = [], []\n    for method, colour in [("baseline", C_BASE), ("framework", C_FRAME)]:\n        data.append(per_run[(per_run.scenario == scenario) &\n                            (per_run.method == method)].nri.to_numpy())\n        colours.append(colour)\n    parts = ax2.violinplot(data, positions=[0, 1], widths=0.75, showextrema=False)\n    for body, colour in zip(parts["bodies"], colours):\n        body.set(facecolor=colour, alpha=0.25, edgecolor=colour, lw=1.1)\n    for i, (v, colour) in enumerate(zip(data, colours)):\n        j = np.random.default_rng(7 + i).uniform(-0.07, 0.07, v.size)\n        ax2.plot(i + j, v, "o", ms=3.0, color=colour, alpha=0.7, mec="none")\n        from scipy import stats as sps\n        h = sps.t.ppf(0.975, v.size - 1) * v.std(ddof=1) / np.sqrt(v.size)\n        ax2.errorbar(i, v.mean(), yerr=h, fmt="D", ms=4.5, color=colour,\n                     mec="white", mew=0.7, elinewidth=1.6, capsize=3.5, zorder=4)\n        ax2.annotate(f"{v.mean():.3f}", xy=(i, v.mean()), xytext=(11, -2),\n                     textcoords="offset points", fontsize=8, color=colour,\n                     fontweight="bold")\n    ax2.set_xticks([0, 1]); ax2.set_xticklabels(["Baseline", "Framework"])\n    ax2.set_ylabel("Normalized resilience index")\n    ax2.set_title("(b) Per-run NRI", loc="left", fontsize=9.5, fontweight="bold")\n    fig.text(0.5, -0.10, "Shaded band: 95% confidence interval of the mean over "\n             "n = 20 runs. Both panels are computed by analysis/calculate_nri.py "\n             "from data/availability_traces/.", ha="center", fontsize=7.4, color="#555")\n    save(fig, out, "figure6_availability_nri")\n\n\ndef figure7(out: Path):\n    fig, ax = plt.subplots(figsize=(5.4, 3.6))\n    r = np.arange(1, 401)\n    for p, colour in zip([0.01, 0.05, 0.10, 0.20],\n                         ["#2F6F8F", "#4E9A6B", "#C8862B", "#B4442E"]):\n        ax.plot(r, audit.p_detect_bound(p, r), color=colour, lw=1.8,\n                label=f"$p$ = {p:.0%} (bound)")\n        exact = [audit.p_detect_exact(10000, int(p * 10000), int(k))\n                 for k in np.linspace(1, 400, 60)]\n        ax.plot(np.linspace(1, 400, 60), exact, ls=":", lw=1.2, color=colour)\n    ax.axhline(0.95, color="#444", ls="--", lw=1.0)\n    ax.annotate("target $\\\\eta$ = 0.95", xy=(300, 0.95), xytext=(0, -12),\n                textcoords="offset points", fontsize=7.6, color="#444")\n    ax.set_xlabel("Number of challenged blocks $r$")\n    ax.set_ylabel("Probability of detecting at least one corrupted block")\n    ax.set_title("Audit-detection sensitivity (l = 10,000 blocks)", loc="left",\n                 fontsize=9.5, fontweight="bold")\n    ax.legend(frameon=False, fontsize=7.8, loc="lower right")\n    ax.set_ylim(0, 1.02); ax.set_xlim(0, 400)\n    fig.text(0.5, -0.06, "Solid: independent-sampling lower bound, Eq. (5). "\n             "Dotted: exact hypergeometric probability, Eq. (4).",\n             ha="center", fontsize=7.4, color="#555")\n    save(fig, out, "figure7_audit_sensitivity")\n\n\ndef figure8(out: Path):\n    """Four-node dependency example of Section 3.5, recomputed from dtcr.risk.\n\n    Panel (b) answers the reviewer request for a spectral radius and a\n    convergence margin. The acyclic chain of Section 3.5 is nilpotent, so\n    rho(lambda W^T) = 0 and Eq. (10) converges for every lambda; the constraint\n    only binds once a feedback edge exists. Both cases are therefore plotted.\n    """\n    names = ["Sensor", "Edge broker", "Analytics", "Civil service"]\n    W = np.zeros((4, 4))\n    W[0, 1] = 0.70   # sensor -> broker\n    W[1, 2] = 0.80   # broker -> analytics\n    W[1, 3] = 0.40   # broker -> civil service\n    W[2, 3] = 0.60   # analytics -> civil service\n    R = np.array([0.60, 0.10, 0.05, 0.02])\n    lam = 0.45\n    Rt = risk.propagate(R, W, lam)\n\n    # Cyclic variant: the civil service feeds a control signal back to the sensor.\n    Wc = W.copy()\n    Wc[3, 0] = 0.55\n\n    fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.7))\n    ax = axes[0]\n    pos = {0: (0.06, 0.52), 1: (0.38, 0.52), 2: (0.70, 0.86), 3: (0.97, 0.30)}\n    label_off = {0: (0, -0.17), 1: (0, -0.17), 2: (-0.20, -0.02), 3: (0, -0.17)}\n    edge_shift = {(0, 1): (0.0, 0.05), (1, 2): (-0.055, 0.02),\n                  (1, 3): (0.0, 0.055), (2, 3): (0.06, 0.02)}\n    for (i, j), w in [((0, 1), 0.70), ((1, 2), 0.80), ((1, 3), 0.40), ((2, 3), 0.60)]:\n        x1, y1 = pos[i]; x2, y2 = pos[j]\n        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),\n                    arrowprops=dict(arrowstyle="-|>", lw=0.7 + 2.2 * w,\n                                    color="#8899A6", shrinkA=18, shrinkB=18))\n        dx, dy = edge_shift[(i, j)]\n        ax.text((x1 + x2) / 2 + dx, (y1 + y2) / 2 + dy, f"{w:.2f}", fontsize=7.8,\n                ha="center", va="center", color="#5A6B77",\n                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))\n    for i, name in enumerate(names):\n        x, y = pos[i]\n        ax.scatter([x], [y], s=300 + 1700 * Rt[i], color=C_FRAME, alpha=0.28,\n                   edgecolors=C_FRAME, linewidths=1.3, zorder=3)\n        ax.text(x, y, f"{Rt[i]:.3f}", ha="center", va="center", fontsize=8,\n                fontweight="bold", color="#12303D", zorder=4)\n        ox, oy = label_off[i]\n        ha = "center" if ox == 0 else ("right" if ox < 0 else "left")\n        va = "center" if ox != 0 else ("top" if oy < 0 else "bottom")\n        ax.text(x + ox, y + oy, f"{name}\\n$R$={R[i]:.2f}", ha=ha, va=va,\n                fontsize=7.6, color="#333")\n    ax.set_xlim(-0.10, 1.14); ax.set_ylim(-0.02, 1.10)\n    ax.axis("off")\n    ax.set_title("(a) Local and propagated risk", loc="left", fontsize=9.5,\n                 fontweight="bold")\n    ax.text(0.0, 0.0, f"$\\\\kappa$ = {risk.amplification(R, Rt):.3f}", fontsize=8.4,\n            color="#12303D", fontweight="bold")\n\n    ax2 = axes[1]\n    lam_star = 1.0 / risk.spectral_radius(Wc, 1.0)   # margin reaches zero here\n    lams = np.linspace(0, lam_star * 1.12, 300)\n    kap_a, kap_c, margin_c = [], [], []\n    for lm in lams:\n        kap_a.append(risk.amplification(R, risk.propagate(R, W, lm)))\n        m = risk.convergence_margin(Wc, lm)\n        margin_c.append(m)\n        kap_c.append(risk.amplification(R, risk.propagate(R, Wc, lm)) if m > 1e-9 else np.nan)\n    l1, = ax2.plot(lams, kap_a, color=C_FRAME, lw=1.9,\n                   label="$\\\\kappa$, acyclic chain (Section 3.5)")\n    l2, = ax2.plot(lams, kap_c, color="#C8862B", lw=1.9,\n                   label="$\\\\kappa$, with feedback edge")\n    ax2.axvline(lam, color="#444", ls="--", lw=1.0)\n    ax2.annotate(f"operating point\\n$\\\\lambda$ = {lam}", xy=(lam, 1.15),\n                 xytext=(6, 0), textcoords="offset points", fontsize=7.6, color="#444")\n    ax2.axvline(lam_star, color=C_BASE, ls="-.", lw=1.0, alpha=0.8)\n    ax2.annotate(f"$\\\\lambda^*$ = {lam_star:.2f}", xy=(lam_star, 1.15), xytext=(-4, 0),\n                 textcoords="offset points", fontsize=7.6, color=C_BASE, ha="right")\n    ax3 = ax2.twinx()\n    l3, = ax3.plot(lams, margin_c, color=C_BASE, lw=1.4, ls=":",\n                   label="$1-\\\\rho(\\\\lambda W^T)$, with feedback edge")\n    ax3.axhline(0, color=C_BASE, lw=0.8, alpha=0.5)\n    ax3.set_ylabel("Convergence margin", color=C_BASE)\n    ax3.tick_params(axis="y", colors=C_BASE); ax3.grid(False)\n    ax2.set_yscale("log")\n    ax2.set_xlabel("Damping factor $\\\\lambda$")\n    ax2.set_ylabel("Aggregate risk amplification $\\\\kappa$ (log scale)")\n    ax2.set_title("(b) Amplification and convergence", loc="left", fontsize=9.5,\n                  fontweight="bold")\n    ax2.legend([l1, l2, l3], [l.get_label() for l in (l1, l2, l3)], frameon=False,\n               fontsize=7.4, loc="upper left")\n    fig.text(0.5, -0.07, "Panel (a) uses the raw edge weights printed in Section 3.5. "\n             "The acyclic chain is nilpotent, so $\\\\rho(\\\\lambda W^T)=0$ and Eq. (10) "\n             "converges for all $\\\\lambda$; the margin is shown for the cyclic variant, "\n             "whose convergence limit is $\\\\lambda^*=1/\\\\rho(W^T)$. The operating point "\n             "$\\\\lambda=0.45$ lies well inside it.",\n             ha="center", fontsize=7.4, color="#555")\n    save(fig, out, "figure8_risk_propagation")\n\n\ndef figure9(integrity: pd.DataFrame, out: Path):\n    d = integrity[(integrity.corruption_fraction != "all")].copy()\n    d["corruption_fraction"] = d.corruption_fraction.astype(float)\n    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.5))\n    ax = axes[0]\n    for s, colour in zip(SCENARIOS, ["#2F6F8F", "#4E9A6B", "#C8862B", "#B4442E"]):\n        sub = d[d.scenario == s].sort_values("corruption_fraction")\n        ax.errorbar(sub.corruption_fraction * 100, sub.recall,\n                    yerr=[sub.recall - sub.recall_ci95_lo,\n                          sub.recall_ci95_hi - sub.recall],\n                    marker="o", ms=4, lw=1.6, capsize=3, color=colour, label=s)\n    ax.set_xlabel("Corrupted fraction of blocks (%)")\n    ax.set_ylabel("Sensitivity (recall)")\n    ax.set_title("(a) Detection sensitivity vs corruption level", loc="left",\n                 fontsize=9.5, fontweight="bold")\n    ax.legend(frameon=False, fontsize=8, ncol=2)\n\n    ax2 = axes[1]\n    pooled = integrity[(integrity.scenario != "pooled") &\n                       (integrity.corruption_fraction == "all")]\n    metrics = ["accuracy", "recall", "specificity", "precision", "f1"]\n    x = np.arange(len(metrics))\n    w = 0.19\n    for i, (s, colour) in enumerate(zip(SCENARIOS,\n                                        ["#2F6F8F", "#4E9A6B", "#C8862B", "#B4442E"])):\n        row = pooled[pooled.scenario == s].iloc[0]\n        ax2.bar(x + (i - 1.5) * w, [row[m] for m in metrics], width=w,\n                color=colour, alpha=0.85, label=s)\n    ax2.set_xticks(x)\n    ax2.set_xticklabels(["Accuracy", "Recall", "Specificity", "Precision", "F1"],\n                        fontsize=8)\n    ax2.set_ylim(0.90, 1.005)\n    ax2.set_ylabel("Value")\n    ax2.set_title("(b) Integrity-verification metrics by scenario", loc="left",\n                  fontsize=9.5, fontweight="bold")\n    ax2.legend(frameon=False, fontsize=8, ncol=4, loc="lower center")\n    fig.text(0.5, -0.06, "Observation unit: one challenged telemetry block. "\n             "Error bars are Wilson 95% intervals.", ha="center", fontsize=7.4,\n             color="#555")\n    save(fig, out, "figure9_integrity_metrics")\n\n\ndef figure10(overhead: pd.DataFrame, out: Path):\n    d = overhead.dropna(subset=["relative_overhead_pct_eq17"])\n    d = d[d.relative_overhead_pct_eq17 > 0]\n    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.4))\n    labels = [m.replace("_", " ") for m in d.metric]\n    y = np.arange(len(d))\n    axes[0].barh(y, d.relative_overhead_pct_eq17, color=C_FRAME, alpha=0.85, height=0.55)\n    axes[0].axvline(6.0, color=C_BASE, ls="--", lw=1.2)\n    axes[0].annotate("published bound 6%", xy=(6.0, len(d) - 0.6), xytext=(4, 0),\n                     textcoords="offset points", fontsize=7.6, color=C_BASE)\n    for i, v in enumerate(d.relative_overhead_pct_eq17):\n        axes[0].text(v + 0.12, i, f"{v:.2f}%", va="center", fontsize=8)\n    axes[0].set_yticks(y); axes[0].set_yticklabels(labels, fontsize=8)\n    axes[0].set_xlabel("Relative overhead vs baseline consumption (%), Eq. (17)")\n    axes[0].set_title("(a) Eq. (17) denominator", loc="left", fontsize=9.5,\n                      fontweight="bold")\n    axes[0].set_xlim(0, 7.5)\n\n    axes[1].barh(y, d.share_of_capacity_pct, color="#4E9A6B", alpha=0.85, height=0.55)\n    for i, v in enumerate(d.share_of_capacity_pct):\n        axes[1].text(v + 0.006, i, f"{v:.3f}%", va="center", fontsize=8)\n    axes[1].set_yticks(y); axes[1].set_yticklabels([])\n    axes[1].set_xlabel("Additional consumption as a share of cluster capacity (%)")\n    axes[1].set_title("(b) Cluster-capacity denominator", loc="left", fontsize=9.5,\n                      fontweight="bold")\n    fig.text(0.5, -0.08, "The two panels use different denominators. Section 3.2 of the "\n             "revised manuscript reports panel (a) and cites panel (b) separately.",\n             ha="center", fontsize=7.4, color="#555")\n    save(fig, out, "figure10_overhead")\n\n\ndef figure11(ablation: pd.DataFrame, out: Path):\n    order = ["B0_ids_manual", "B1_ids_playbook", "B2_stack_no_dt",\n             "A1_no_graph", "A2_no_whatif", "FULL_framework"]\n    d = ablation.set_index("variant").loc[order].reset_index()\n    short = [v.split("_", 1)[0] for v in d.variant]\n    fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.5))\n    x = np.arange(len(d))\n\n    axes[0].bar(x - 0.2, d.detection_latency_mean_s, 0.4, yerr=d.detection_latency_sd_s,\n                color=C_BASE, alpha=0.85, capsize=3, label="Detection")\n    axes[0].bar(x + 0.2, d.recovery_time_mean_s / 10, 0.4,\n                yerr=d.recovery_time_sd_s / 10, color=C_FRAME, alpha=0.85,\n                capsize=3, label="Recovery / 10")\n    axes[0].set_xticks(x); axes[0].set_xticklabels(short, fontsize=8)\n    axes[0].set_ylabel("Seconds")\n    axes[0].set_title("(a) Latency", loc="left", fontsize=9.5, fontweight="bold")\n    axes[0].legend(frameon=False, fontsize=8)\n\n    for col, colour, label in [("unsafe_action_rate", "#B4442E", "Unsafe action"),\n                               ("policy_violation_rate", "#C8862B", "Policy violation"),\n                               ("rollback_rate", "#2F6F8F", "Rollback")]:\n        axes[1].plot(x, d[col], marker="o", ms=4.5, lw=1.7, color=colour, label=label)\n        axes[1].fill_between(x, d[f"{col}_ci95_lo"], d[f"{col}_ci95_hi"],\n                             color=colour, alpha=0.15, lw=0)\n    axes[1].set_xticks(x); axes[1].set_xticklabels(short, fontsize=8)\n    axes[1].set_ylabel("Rate")\n    axes[1].set_title("(b) Safety of automated action", loc="left", fontsize=9.5,\n                      fontweight="bold")\n    axes[1].legend(frameon=False, fontsize=8)\n\n    axes[2].plot(x, d.recovery_success_rate, marker="s", ms=4.5, lw=1.7,\n                 color="#4E9A6B", label="Recovery success")\n    axes[2].fill_between(x, d.recovery_success_rate_ci95_lo,\n                         d.recovery_success_rate_ci95_hi, color="#4E9A6B",\n                         alpha=0.15, lw=0)\n    axes[2].plot(x, d.risk_ranking_accuracy, marker="^", ms=4.5, lw=1.7,\n                 color="#2F6F8F", label="Risk-ranking accuracy")\n    axes[2].fill_between(x, d.risk_ranking_accuracy_ci95_lo,\n                         d.risk_ranking_accuracy_ci95_hi, color="#2F6F8F",\n                         alpha=0.15, lw=0)\n    axes[2].set_xticks(x); axes[2].set_xticklabels(short, fontsize=8)\n    axes[2].set_ylabel("Rate"); axes[2].set_ylim(0.5, 1.02)\n    axes[2].set_title("(c) Correctness", loc="left", fontsize=9.5, fontweight="bold")\n    axes[2].legend(frameon=False, fontsize=8, loc="lower left")\n    fig.text(0.5, -0.07, "B0 IDS+manual; B1 IDS+playbook; B2 security stack without the "\n             "digital twin; A1 framework without graph propagation; A2 framework without "\n             "what-if simulation; FULL complete framework. Bands are Wilson 95% intervals.",\n             ha="center", fontsize=7.4, color="#555")\n    save(fig, out, "figure11_ablation")\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=__doc__,\n                                 formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--results", default="results")\n    ap.add_argument("--configs", default="configs/framework_parameters.yaml")\n    ap.add_argument("--out", default="figures")\n    args = ap.parse_args()\n    data, res, out = Path(args.data), Path(args.results), Path(args.out)\n\n    cfg = load_config(Path(args.configs))\n    runs = pd.read_csv(data / "run_level_metrics.csv")\n    traj = pd.read_csv(res / "availability_trajectories.csv")\n    per_run = pd.read_csv(res / "nri_per_run.csv")\n    integrity = pd.read_csv(res / "table_S4_integrity.csv")\n    overhead = pd.read_csv(res / "table_S5_overhead.csv")\n    ablation = pd.read_csv(res / "table_S6_ablation.csv")\n\n    print("generating figures:")\n    figure5(runs, out)\n    figure6(traj, per_run, cfg, out)\n    figure7(out)\n    figure8(out)\n    figure9(integrity, out)\n    figure10(overhead, out)\n    figure11(ablation, out)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
    SCRIPTS['verify_repository.py'] = '#!/usr/bin/env python3\n"""Check the deposit for internal consistency and submission readiness.\n\nTwo classes of check are run.\n\nSTRUCTURAL / CONSISTENCY (must pass at all times):\n  * every file promised by the README exists;\n  * run_level_metrics.csv has the declared design (scenarios x arms x reps);\n  * every run references an availability trace that exists and parses;\n  * recovery time and NRI recomputed from the traces match the stored values;\n  * the aggregates in results/summary.json match the values quoted in the\n    manuscript to the precision at which the manuscript prints them.\n\nSUBMISSION READINESS (expected to fail while the data are synthetic):\n  * no file carries data_origin = synthetic_reference;\n  * the Zenodo DOI placeholder has been replaced in README and CITATION.cff.\n\nExit status is 0 when the structural checks pass, 1 otherwise.  ``--strict``\nadditionally requires the submission-readiness checks to pass.\n\nUsage:  python analysis/verify_repository.py --root . [--strict]\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nREQUIRED = [\n    "README.md", "LICENSE", "LICENSE-DATA", "CITATION.cff", "PROVENANCE.md",\n    "DATA_DICTIONARY.md", "PROTOCOL.md", "THREAT_MODEL.md", "Makefile",\n    "data/run_level_metrics.csv", "data/ablation_runs.csv",\n    "data/confusion_matrices/integrity_confusion.csv",\n    "data/resource_measurements/resource_usage.csv",\n    "configs/framework_parameters.yaml",\n    "analysis/statistics.py", "analysis/calculate_nri.py",\n    "analysis/generate_figures.py", "analysis/simulate_reference_dataset.py",\n    "environment/requirements.txt", "environment/software_versions.md",\n]\n\n# Values printed in the manuscript, and the tolerance implied by their precision.\nMANUSCRIPT_CLAIMS = {\n    "detection_latency_baseline_s": (43.1, 0.05),\n    "detection_latency_framework_s": (8.5, 0.05),\n    "recovery_time_baseline_s": (399.0, 0.5),\n    "recovery_time_framework_s": (122.0, 0.5),\n    "nri_s3_baseline": (0.71, 0.005),\n    "nri_s3_framework": (0.93, 0.005),\n    "integrity_accuracy": (0.987, 0.0005),\n}\n\nPLACEHOLDER_DOI = "10.5281/zenodo.PLACEHOLDER"\n\n\nclass Report:\n    def __init__(self):\n        self.structural, self.readiness = [], []\n\n    def add(self, group, ok, name, detail=""):\n        group.append({"ok": bool(ok), "check": name, "detail": detail})\n\n    def render(self) -> str:\n        lines = []\n        for title, group in [("STRUCTURAL / CONSISTENCY", self.structural),\n                             ("SUBMISSION READINESS", self.readiness)]:\n            lines.append(f"\\n{title}")\n            lines.append("-" * len(title))\n            for c in group:\n                mark = "PASS" if c["ok"] else "FAIL"\n                lines.append(f"  [{mark}] {c[\'check\']}"\n                             + (f"\\n         {c[\'detail\']}" if c["detail"] else ""))\n        return "\\n".join(lines)\n\n\ndef check_files(root: Path, rep: Report):\n    missing = [f for f in REQUIRED if not (root / f).exists()]\n    rep.add(rep.structural, not missing, f"{len(REQUIRED)} required files present",\n            "missing: " + ", ".join(missing) if missing else "")\n\n\ndef check_design(root: Path, rep: Report, runs: pd.DataFrame):\n    scen = sorted(runs.scenario.unique())\n    arms = sorted(runs.method.unique())\n    counts = runs.groupby(["scenario", "method"]).size()\n    ok = (scen == ["S1", "S2", "S3", "S4"] and arms == ["baseline", "framework"]\n          and counts.nunique() == 1)\n    rep.add(rep.structural, ok,\n            f"experimental design: {len(scen)} scenarios x {len(arms)} arms "\n            f"x {counts.iloc[0]} repetitions = {len(runs)} runs",\n            "" if ok else f"unbalanced cells: {counts.to_dict()}")\n    cens = int(runs.recovery_censored.sum()) if "recovery_censored" in runs else 0\n    rep.add(rep.structural, True, f"censored runs recorded: {cens}",\n            "censored runs are reported in the run count and excluded from means")\n\n\ndef check_traces(root: Path, rep: Report, runs: pd.DataFrame):\n    missing, unreadable = [], []\n    for _, r in runs.iterrows():\n        p = root / "data" / r.trace_file\n        if not p.exists():\n            missing.append(r.run_id)\n            continue\n        try:\n            df = pd.read_csv(p)\n            if not {"t_s", "availability"} <= set(df.columns) or df.empty:\n                unreadable.append(r.run_id)\n        except Exception:\n            unreadable.append(r.run_id)\n    ok = not missing and not unreadable\n    rep.add(rep.structural, ok, f"{len(runs)} availability traces present and parseable",\n            "" if ok else f"missing={missing[:5]} unreadable={unreadable[:5]}")\n\n\ndef check_recomputation(root: Path, rep: Report):\n    p = root / "results" / "nri_parameters.json"\n    if not p.exists():\n        rep.add(rep.structural, False, "NRI recomputation available",\n                "run `make analysis` first")\n        return\n    c = json.loads(p.read_text())["consistency_check"]\n    ok = c["mismatched_runs"] == 0\n    rep.add(rep.structural, ok,\n            "recovery time and NRI recomputed from traces match stored values",\n            f"max |dNRI| = {c[\'max_abs_delta_nri\']:.2e}, "\n            f"max |drecovery| = {c[\'max_abs_delta_recovery_s\']:.2e}, "\n            f"mismatched = {c[\'mismatched_runs\']}")\n\n\ndef check_claims(root: Path, rep: Report):\n    p = root / "results" / "summary.json"\n    if not p.exists():\n        rep.add(rep.structural, False, "manuscript claims reproduced",\n                "run `make analysis` first")\n        return\n    s = json.loads(p.read_text())\n    actual = {\n        "detection_latency_baseline_s": s["detection_latency_s"]["baseline_mean"],\n        "detection_latency_framework_s": s["detection_latency_s"]["framework_mean"],\n        "recovery_time_baseline_s": s["recovery_time_s"]["baseline_mean"],\n        "recovery_time_framework_s": s["recovery_time_s"]["framework_mean"],\n        "nri_s3_baseline": s["nri_S3"]["baseline_mean"],\n        "nri_s3_framework": s["nri_S3"]["framework_mean"],\n        "integrity_accuracy": s["integrity_pooled"]["accuracy"],\n    }\n    bad = []\n    for k, (claim, tol) in MANUSCRIPT_CLAIMS.items():\n        if abs(actual[k] - claim) > tol:\n            bad.append(f"{k}: manuscript {claim}, data {actual[k]:.4f}")\n    rep.add(rep.structural, not bad,\n            f"{len(MANUSCRIPT_CLAIMS)} manuscript claims reproduced from data/",\n            "; ".join(bad))\n    ovh = s["overhead_max_relative_pct_eq17"]\n    rep.add(rep.structural, ovh < 6.0,\n            f"overhead below the published 6% bound (max {ovh:.2f}%)")\n\n\ndef check_synthetic(root: Path, rep: Report):\n    hits = []\n    for p in sorted(root.rglob("*.csv")):\n        try:\n            head = p.open(encoding="utf-8", errors="ignore").read(4096)\n        except OSError:\n            continue\n        if "synthetic_reference" in head:\n            hits.append(str(p.relative_to(root)))\n    rep.add(rep.readiness, not hits,\n            "no file is marked data_origin = synthetic_reference",\n            f"{len(hits)} file(s) still synthetic, e.g. {hits[:3]}" if hits else "")\n\n\ndef check_doi(root: Path, rep: Report):\n    hits = [f for f in ("README.md", "CITATION.cff", "PROVENANCE.md")\n            if (root / f).exists() and PLACEHOLDER_DOI in (root / f).read_text()]\n    rep.add(rep.readiness, not hits,\n            "Zenodo DOI placeholder replaced with the minted DOI",\n            f"placeholder still present in: {\', \'.join(hits)}" if hits else "")\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=__doc__,\n                                 formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--root", default=".")\n    ap.add_argument("--strict", action="store_true",\n                    help="also require the submission-readiness checks to pass")\n    args = ap.parse_args()\n    root = Path(args.root).resolve()\n    rep = Report()\n\n    check_files(root, rep)\n    runs_path = root / "data" / "run_level_metrics.csv"\n    if runs_path.exists():\n        runs = pd.read_csv(runs_path)\n        check_design(root, rep, runs)\n        check_traces(root, rep, runs)\n    else:\n        rep.add(rep.structural, False, "run_level_metrics.csv readable")\n    check_recomputation(root, rep)\n    check_claims(root, rep)\n    check_synthetic(root, rep)\n    check_doi(root, rep)\n\n    print(rep.render())\n    struct_ok = all(c["ok"] for c in rep.structural)\n    ready_ok = all(c["ok"] for c in rep.readiness)\n    print(f"\\nstructural: {\'PASS\' if struct_ok else \'FAIL\'}   "\n          f"submission-ready: {\'YES\' if ready_ok else \'NO\'}")\n    if not ready_ok:\n        print("\\nThe deposit is internally consistent but still carries the synthetic\\n"\n              "reference dataset. See PROVENANCE.md before citing it in a submission.")\n    if not struct_ok:\n        return 1\n    return 1 if (args.strict and not ready_ok) else 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
    CONFIG = '# Actual experimental parameters of the DTCR framework.\n#\n# This file replaces the "illustrative value" column of manuscript Table 1.\n# Every value here is the value the analysis code actually consumes; nothing in\n# this file is illustrative. Manuscript Table 1 is retained for symbol\n# definitions only, and the revised manuscript cites this file as the source of\n# the operating point.\n\nmetadata:\n  release: "v1.0.0"\n  parameter_set_id: "dtcr-op-2026-07"\n  applies_to_scenarios: [S1, S2, S3, S4]\n\ndigital_twin:\n  epsilon_sync: 0.08          # tolerance of Eq. (1); above it an asset is stale\n  epsilon_div_guard: 1.0e-6   # epsilon of Eq. (1), prevents division by zero\n  sync_period_s: 1.0\n  stale_asset_policy: "exclude_from_autonomous_recovery"\n\ntrust:                        # Eq. (2)-(3)\n  alpha: 0.40                 # integrity component c_i\n  beta: 0.35                  # provenance component q_i\n  gamma: 0.25                 # behavioural component b_i\n  rho: 0.60                   # temporal memory of Eq. (3)\n  initial_trust: 1.00\n  update_period_s: 5.0\n\nintegrity_audit:              # Eq. (4)-(5)\n  block_size_bytes: 4096\n  blocks_per_replica_l: 10000\n  challenged_blocks_r: 59\n  target_detection_probability_eta: 0.95\n  assumed_corruption_fraction: 0.05\n  audit_cadence_s: 30\n  hash: "BLAKE2b-256 over 4 KiB blocks; homomorphic tag over the prime-order group of Curve25519"\n  nonce_bits: 128\n  challenge_rng: "HMAC-DRBG(SHA-256), seeded per audit round from the verifier\'s HSM"\n\nanomaly:                      # Eq. (6)-(7)\n  feature_vector_dimension_p: 9\n  features:\n    - mqtt_messages_per_s\n    - mqtt_interarrival_cv\n    - tcp_syn_rate_per_s\n    - tcp_established_connections\n    - cpu_utilisation_pct\n    - resident_memory_mb\n    - egress_kbps\n    - control_command_rate_per_min\n    - provenance_hop_count\n  normal_window_s: 1800       # baseline estimation window\n  calibration_window_s: 900   # held out from the baseline window\n  test_separation: "baseline / calibration / evaluation are disjoint in time"\n  covariance_shrinkage: 0.05  # Sigma + shrinkage * tr(Sigma)/p * I\n  score_mapping: "chi2_cdf"   # corrected Eq. (7); "legacy_exp" reproduces the printed form\n  gaussian_check: "Kolmogorov-Smirnov against chi2_p on the calibration window"\n\nrisk:                         # Eq. (8)-(11)\n  local_aggregation: "additive"   # selected by AUC; "product" is the printed Eq. (8)\n  additive_weights: {w_a: 0.45, w_t: 0.35, w_at: 0.20}\n  criticality_s:\n    sensor: 0.30\n    edge_broker: 0.65\n    analytics: 0.75\n    civil_service: 1.00\n  lambda: 0.45                # damping factor\n  W_normalisation: "column-stochastic on outgoing influence mass"\n  theta: 0.35                 # orchestration trigger threshold on the propagated score\n  theta_domain_note: >\n    After column normalisation and with lambda = 0.45 the propagated score is\n    bounded by 1/(1-lambda) = 1.818, so theta is defined on [0, 1.818] and not\n    on [0, 1]. The propagated score is an exposure index, not a probability.\n\norchestration:                # Eq. (12)-(13)\n  mu1_cpu: 0.20\n  mu2_net: 0.15\n  mu3_disruption: 0.25\n  policy_violation: "hard constraint (candidate rejected), not a finite penalty"\n  tau_min_host_trust:\n    sensor: 0.50\n    edge_broker: 0.70\n    analytics: 0.75\n    civil_service: 0.85\n  action_simulation_horizon_s: 120\n  orchestration_cycle_period_s: 5\n  candidate_actions: [isolate, rate_limit, revoke_identity, migrate, restart, restore_validated_state]\n  solver: "exhaustive enumeration over the affected subgraph (<= 64 candidates)"\n  solver_tolerance: 1.0e-9\n  solver_timeout_s: 2.0\n  tie_breaking: "lowest disruption, then lowest compute overhead, then action name"\n  revalidation: "policy re-checked against live state immediately before enforcement"\n  rollback: "automatic on failed recovery validation within 2 orchestration cycles"\n\nresilience:                   # Eq. (15), (18)-(19)\n  rto_s: 300.0\n  a_min: 0.95\n  a_max: 1.00\n  hold_interval_s: 30.0\n  sampling_interval_s: 1.0\n  integration_window: "[t_dis, t_dis + 2*RTO]"\n\nexperiment:\n  repetitions_per_cell: 20\n  scenarios: [S1, S2, S3, S4]\n  arms: [baseline, framework]\n  schedule: "interleaved matched pairs; arms alternate within a repetition block"\n  randomisation_seed: 20260731\n  washout_s: 300\n  reset: "full redeploy of the edge namespace and twin state between repetitions"\n  censoring_rule: >\n    A run whose availability never satisfies the hold condition inside the\n    observation window is recorded with recovery_censored = 1 and excluded from\n    the mean while being reported in the run count.\n'
    for name, text in SCRIPTS.items():
        (ROOT/'analysis'/name).write_text(text)
    (ROOT/'configs'/'framework_parameters.yaml').write_text(CONFIG)
    print('wrote', len(SCRIPTS), 'analysis scripts + config')
else:
    print('cloning; skipping embedded scripts')

## 4. Run the pipeline

Reference dataset → statistics → NRI recomputation → figures → verification.
This is exactly what `make all` does in the deposit.

In [ ]:
os.chdir(ROOT)
def run(mod, *args):
    cmd = [sys.executable, f"analysis/{mod}", *args]
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode:
        print("STDERR", r.stderr[-1500:])
    return r.returncode

run("simulate_reference_dataset.py", "--out", "data")
run("statistics.py", "--data", "data", "--out", "results")
run("calculate_nri.py", "--data", "data", "--out", "results", "--strict")
run("generate_figures.py", "--data", "data", "--results", "results", "--out", "figures")
run("verify_repository.py", "--root", ".")

## 5. The algorithm, interactively

The reference library is now importable. These cells exercise each block of the
mathematical model on its worked example.

In [ ]:
sys.path.insert(0, str((ROOT/"analysis").resolve()))
from dtcr import audit, trust, anomaly, risk, orchestration, resilience, stats
import numpy as np

# Probabilistic block audit (Eq. 4-5)
print("Audit: r_min(5% corruption, 95% target) =", audit.r_min(0.05, 0.95),
      "| exact P_det =", round(audit.p_detect_exact(10000, 500, 59), 4))

# Dynamic trust (Eq. 2-3)
tt = trust.TrustTracker(initial=0.92)
print("Trust after one degraded window:", round(tt.update(0.95, 0.90, 0.80), 4))

# Dependency-risk propagation (Eq. 8-11)
W = np.zeros((4, 4)); W[0,1], W[1,2], W[1,3], W[2,3] = 0.70, 0.80, 0.40, 0.60
R = np.array([0.60, 0.10, 0.05, 0.02])
Rt = risk.propagate(R, W, 0.45)
print("Propagated risk:", np.round(Rt, 3),
      "| amplification kappa =", round(risk.amplification(R, Rt), 3))

In [ ]:
# Policy-constrained orchestration (Eq. 12-13, Algorithm 1)
nodes = {
  "cloud-01": orchestration.Node("cloud-01",
      orchestration.ResourceVector(8, 16000, 200000, 1000000),
      security_label=3, trust=0.9, domain="d0"),
  "edge-04": orchestration.Node("edge-04",
      orchestration.ResourceVector(4, 8000, 60000, 100000),
      security_label=1, trust=0.6, domain="d2"),
}
workloads = {"analytics-core": orchestration.Workload("analytics-core",
      orchestration.ResourceVector(2, 4000, 20000, 40000),
      security_label=3, min_host_trust=0.75, allowed_domains=("d0",))}
cands = [
  orchestration.Candidate("migrate_cloud", 0.31, 0.42, 0.20, 0.30, {"analytics-core": "cloud-01"}),
  orchestration.Candidate("migrate_edge",  0.10, 0.05, 0.05, 0.05, {"analytics-core": "edge-04"}),
]
res = orchestration.select_action(cands, nodes, workloads)
print("Selected action:", res["selected"].action)
print("Rejected (inadmissible):", [(r["action"], r["reasons"]) for r in res["rejected"]])

## 6. Visualizations

Every manuscript figure, rendered inline from the pipeline output.

In [ ]:
from IPython.display import Image, display
import glob, os
for f in sorted(glob.glob("figures/*.png")):
    print("\n" + "=" * 80 + "\n" + os.path.basename(f))
    display(Image(filename=f, width=900))

## 7. Report

A self-contained summary assembled from `results/summary.json` and the NRI
consistency check.

In [ ]:
import json
s = json.load(open("results/summary.json"))
nri = json.load(open("results/nri_parameters.json"))
def fmt(x, n=1):
    try: return f"{float(x):.{n}f}"
    except Exception: return str(x)

d, r, g, ig = s["detection_latency_s"], s["recovery_time_s"], s["nri_S3"], s["integrity_pooled"]
print(f"DTCR reproducibility report  (data_origin = {s['data_origin']})")
print("=" * 64)
print(f"Design: 4 scenarios x 2 arms x {s['n_per_cell']} paired repetitions\n")
print(f"Detection latency  : {fmt(d['baseline_mean'])} -> {fmt(d['framework_mean'])} s "
      f"({fmt(d['relative_reduction_pct'])}% reduction, Hedges g={fmt(d['hedges_g'],2)}, p={d['p_value']:.1e})")
print(f"  baseline 95% CI  : [{fmt(d['baseline_ci95'][0])}, {fmt(d['baseline_ci95'][1])}]")
print(f"  framework 95% CI : [{fmt(d['framework_ci95'][0])}, {fmt(d['framework_ci95'][1])}]")
print(f"Recovery time      : {fmt(r['baseline_mean'])} -> {fmt(r['framework_mean'])} s "
      f"({fmt(r['relative_reduction_pct'])}% reduction, Hedges g={fmt(r['hedges_g'],2)})")
print(f"NRI (S3)           : {fmt(g['baseline_mean'],3)} -> {fmt(g['framework_mean'],3)} "
      f"(deficit -{fmt(g['deficit_reduction_pct'])}%)")
print(f"Integrity (pooled) : accuracy {fmt(ig['accuracy'],4)} "
      f"[{fmt(ig['accuracy_ci95'][0],4)}, {fmt(ig['accuracy_ci95'][1],4)}], "
      f"F1 {fmt(ig['f1'],4)}, n={ig['n']} blocks")
print(f"Overhead (max)     : {fmt(s['overhead_max_relative_pct_eq17'],2)}% (Eq.17), < 6% bound")
print(f"NRI consistency    : max|dNRI|={nri['consistency_check']['max_abs_delta_nri']:.2e}, "
      f"mismatched runs={nri['consistency_check']['mismatched_runs']}")
print("\nFigure 5 (latency) and Figure 6 (NRI) are computed from a single source,")
print("so they cannot disagree. Replace data/ with real exports and re-run to")
print("reproduce the real Results section unchanged.")

## 8. Next steps

- Push the deposit to GitHub and set `GIT_URL` at the top to run against the real repository.
- Replace `data/` with real measurement exports (schema in `DATA_DICTIONARY.md`),
  delete `data/generation_manifest.json`, and re-run.
- Mint a Zenodo DOI and set it in `README.md` and `CITATION.cff`; `make verify`
  then reports **submission-ready: YES**.